## Reading pkl file instead

In [9]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from pathlib import Path
from scipy.ndimage import map_coordinates


# ==================================================
# Settings
# ==================================================
PKL_FILE = r"C:\Users\domin\Downloads\processed_data_full_05_19_2026.pkl"

OUT_DIR = Path(r"BLO_perturbation_keogram_outputs_v3\May_19_2026")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX_KM = 1.0
DY_KM = 1.0

DT_SECONDS = 120

FPS = 20
DPI = 200

CMAP = "RdBu_r"

# Make keograms every 5 degrees
KEOGRAM_ANGLES_DEG = np.arange(0, 176, 5)

# Only draw every 15 degrees on the first frame
PLOT_CUT_ANGLES_DEG = np.arange(0, 176, 15)

# Average a wider strip to reduce noise
KEOGRAM_WIDTH_PIXELS = 7

# Keogram display limits
KEOGRAM_TIME_MIN_MAX = 270
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250

# Distance rows to plot from each keogram
PROFILE_DISTANCES_KM = [-250, 0, 250]

# Percentile used to determine the color range separately
# for each directional keogram.
KEOGRAM_COLOR_PERCENTILE = 99

# Fractional padding added to the automatically determined
# y-range of each profile plot.
PROFILE_Y_PADDING_FRACTION = 0.05


# ==================================================
# Load PKL perturbation array
# ==================================================
with open(PKL_FILE, "rb") as file:
    data = pickle.load(file)

# PKL format:
# data[5] = ["Perturbation Images", perturbation_array]
# data[7] = ["Datetime", [date, start_time]]
wave_cube = np.asarray(data[5][1], dtype=np.float32)
date_start = data[7][1]

# Expected shape: time, y, x
if wave_cube.ndim != 3:
    raise ValueError(
        f"Expected perturbation array with shape (time, y, x), "
        f"got {wave_cube.shape}"
    )

nt, NY, NX = wave_cube.shape

print("Perturbation array shape:", wave_cube.shape)
print("Date/start time:", date_start)


# ==================================================
# Coordinate grid
# ==================================================
x_km = np.arange(NX) * DX_KM
y_km = np.arange(NY) * DY_KM

times_s = np.arange(nt) * DT_SECONDS
times_min = times_s / 60.0


# ==================================================
# Color limits
# ==================================================
vmax = np.nanpercentile(np.abs(wave_cube), 99)

if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

vmin = -vmax


# ==================================================
# Animation
# ==================================================
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        x_km.min(),
        x_km.max(),
        y_km.max(),
        y_km.min(),
    ],
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04,
)
cbar.set_label("Perturbation amplitude")

title = ax.set_title("")

ax.set_xlabel("E-W distance (km)")
ax.set_ylabel("N-S distance (km)")
ax.set_aspect("equal")


def update(frame):
    im.set_data(wave_cube[frame])

    title.set_text(
        f"BLO Perturbation Image | "
        f"Frame {frame}/{nt - 1} | "
        f"t = {times_min[frame]:.1f} min"
    )

    return im, title


ani = FuncAnimation(
    fig,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

out_mp4 = OUT_DIR / "BLO_perturbation_animation.mp4"

writer = FFMpegWriter(
    fps=FPS,
    bitrate=3000,
)

ani.save(
    out_mp4,
    writer=writer,
    dpi=DPI,
)

plt.close(fig)


# ==================================================
# Save first frame with directional keogram cuts
# Only show every 15 degrees, but still create
# keograms every 5 degrees later
# ==================================================
fig_first, ax_first = plt.subplots(figsize=(7, 6))

im_first = ax_first.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        x_km.min(),
        x_km.max(),
        y_km.max(),
        y_km.min(),
    ],
)

cbar_first = fig_first.colorbar(
    im_first,
    ax=ax_first,
    fraction=0.046,
    pad=0.04,
)
cbar_first.set_label("Perturbation amplitude")

cx_km = 0.5 * (NX - 1) * DX_KM
cy_km = 0.5 * (NY - 1) * DY_KM

half_length_km = 0.5 * min(
    (NX - 1) * DX_KM,
    (NY - 1) * DY_KM,
)

for angle_deg in PLOT_CUT_ANGLES_DEG:
    theta_cut = np.deg2rad(angle_deg)

    dx_line = half_length_km * np.cos(theta_cut)
    dy_line = half_length_km * np.sin(theta_cut)

    x_line = [
        cx_km - dx_line,
        cx_km + dx_line,
    ]

    y_line = [
        cy_km - dy_line,
        cy_km + dy_line,
    ]

    ax_first.plot(
        x_line,
        y_line,
        linewidth=1.4,
        alpha=0.85,
        label=f"{angle_deg:.0f}°",
    )

ax_first.set_title(
    "First Perturbation Frame with Directional Keogram Cuts"
)
ax_first.set_xlabel("E-W distance (km)")
ax_first.set_ylabel("N-S distance (km)")
ax_first.set_aspect("equal")

ax_first.legend(
    loc="upper right",
    fontsize=7,
    ncol=2,
    framealpha=0.8,
)

plt.tight_layout()

out_first = (
    OUT_DIR
    / "first_perturbation_frame_with_keogram_cuts_every_15deg.png"
)

fig_first.savefig(
    out_first,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_first)


# ==================================================
# Directional keogram helper
# ==================================================
def make_directional_keogram(
    wave_cube,
    angle_deg,
    width_pixels=3,
):
    """
    Create a directional keogram through the center
    of the image cube.

    angle_deg:
        0 degrees  = E-W slice
        90 degrees = N-S slice
    """

    nt, NY, NX = wave_cube.shape

    theta = np.deg2rad(angle_deg)

    # Unit vector along the keogram direction
    ux = np.cos(theta)
    uy = np.sin(theta)

    # Unit vector perpendicular to the keogram direction
    px = -np.sin(theta)
    py = np.cos(theta)

    cx = (NX - 1) / 2
    cy = (NY - 1) / 2

    half_length_pix = min(NX, NY) // 2

    distances_pix = np.arange(
        -half_length_pix,
        half_length_pix + 1,
    )

    offsets = (
        np.arange(width_pixels)
        - width_pixels // 2
    )

    keogram = np.full(
        (nt, len(distances_pix)),
        np.nan,
        dtype=float,
    )

    for it in range(nt):
        sampled_slices = []

        for off in offsets:
            x_sample = (
                cx
                + distances_pix * ux
                + off * px
            )

            y_sample = (
                cy
                + distances_pix * uy
                + off * py
            )

            valid = (
                (x_sample >= 0)
                & (x_sample <= NX - 1)
                & (y_sample >= 0)
                & (y_sample <= NY - 1)
            )

            values = np.full(
                len(distances_pix),
                np.nan,
            )

            values[valid] = map_coordinates(
                wave_cube[it],
                [
                    y_sample[valid],
                    x_sample[valid],
                ],
                order=1,
                mode="nearest",
            )

            sampled_slices.append(values)

        keogram[it] = np.nanmean(
            sampled_slices,
            axis=0,
        )

    # This assumes square 1-km pixels.
    # Because DX_KM = DY_KM = 1 km here,
    # one pixel along any angle equals 1 km.
    distance_km = distances_pix * DX_KM

    return keogram, distance_km


# ==================================================
# Create and save directional keograms
#
# For each keogram:
#   1. Save the keogram image
#   2. Save the full keogram array
#   3. Plot the -250, 0, and +250 km time series
#      together on one plot
# ==================================================
saved_keograms = []
saved_profile_plots = []

for angle_deg in KEOGRAM_ANGLES_DEG:

    # ----------------------------------------------
    # Make keogram
    # ----------------------------------------------
    keogram, distance_km = make_directional_keogram(
        wave_cube,
        angle_deg,
        width_pixels=KEOGRAM_WIDTH_PIXELS,
    )

    # Determine a separate symmetric color range for this keogram,
    # using only the time/distance region that will be displayed.
    visible_time_mask = (
        (times_min >= 0)
        & (times_min <= KEOGRAM_TIME_MIN_MAX)
    )
    visible_distance_mask = (
        (distance_km >= KEOGRAM_DISTANCE_MIN)
        & (distance_km <= KEOGRAM_DISTANCE_MAX)
    )

    visible_keogram = keogram[
        np.ix_(visible_time_mask, visible_distance_mask)
    ]
    finite_keogram_values = visible_keogram[
        np.isfinite(visible_keogram)
    ]

    if finite_keogram_values.size > 0:
        keogram_vmax = np.nanpercentile(
            np.abs(finite_keogram_values),
            KEOGRAM_COLOR_PERCENTILE,
        )
    else:
        keogram_vmax = 1.0

    if not np.isfinite(keogram_vmax) or keogram_vmax == 0:
        keogram_vmax = 1.0

    keogram_vmin = -keogram_vmax

    # ----------------------------------------------
    # Plot keogram
    # ----------------------------------------------
    fig_k, ax_k = plt.subplots(figsize=(9, 5))

    im_k = ax_k.imshow(
        keogram.T,
        origin="lower",
        aspect="auto",
        cmap=CMAP,
        vmin=keogram_vmin,
        vmax=keogram_vmax,
        extent=[
            times_min.min(),
            times_min.max(),
            distance_km.min(),
            distance_km.max(),
        ],
    )

    cbar_k = fig_k.colorbar(
        im_k,
        ax=ax_k,
        fraction=0.046,
        pad=0.04,
    )
    cbar_k.set_label("Perturbation amplitude")

    ax_k.set_title(
        f"BLO Directional Keogram: {angle_deg:.0f}°"
    )
    ax_k.set_xlabel("Time (min)")
    ax_k.set_ylabel("Distance along slice (km)")

    # Keep the requested time and distance display ranges.
    ax_k.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
    ax_k.set_ylim(
        KEOGRAM_DISTANCE_MIN,
        KEOGRAM_DISTANCE_MAX,
    )

    # Show the three selected rows on the keogram
    for requested_distance in PROFILE_DISTANCES_KM:
        distance_index = np.argmin(
            np.abs(distance_km - requested_distance)
        )

        actual_distance = distance_km[distance_index]

        ax_k.axhline(
            actual_distance,
            linewidth=1.0,
            linestyle="--",
            alpha=0.8,
        )

    plt.tight_layout()

    out_png = (
        OUT_DIR
        / f"BLO_keogram_angle_{angle_deg:03.0f}.png"
    )

    out_npy = (
        OUT_DIR
        / f"BLO_keogram_angle_{angle_deg:03.0f}.npy"
    )

    fig_k.savefig(
        out_png,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig_k)

    np.save(
        out_npy,
        keogram,
    )

    saved_keograms.append(out_png)

    # ----------------------------------------------
    # Plot -250, 0, and +250 km rows
    # on the same time-series plot
    # ----------------------------------------------
    fig_profile, ax_profile = plt.subplots(
        figsize=(10, 5)
    )

    profile_data = {}

    for requested_distance in PROFILE_DISTANCES_KM:

        # Find nearest available keogram distance
        distance_index = np.argmin(
            np.abs(distance_km - requested_distance)
        )

        actual_distance = distance_km[distance_index]

        # Keogram shape is:
        # time x distance
        profile = keogram[:, distance_index]

        profile_data[
            f"{actual_distance:.1f}_km"
        ] = profile

        ax_profile.plot(
            times_min,
            profile,
            linewidth=1.3,
            label=(
                f"{actual_distance:+.1f} km"
            ),
        )

    ax_profile.axhline(
        0,
        linewidth=0.8,
        linestyle="--",
        alpha=0.7,
    )

    ax_profile.set_title(
        f"BLO Keogram Profiles at "
        f"-250, 0, and +250 km: {angle_deg:.0f}°"
    )

    ax_profile.set_xlabel("Time (min)")
    ax_profile.set_ylabel("Perturbation amplitude")

    # Keep the requested time range and calculate the amplitude
    # range independently for this profile figure.
    ax_profile.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

    finite_profile_parts = [
        profile[
            visible_time_mask & np.isfinite(profile)
        ]
        for profile in profile_data.values()
        if np.any(
            visible_time_mask & np.isfinite(profile)
        )
    ]

    if finite_profile_parts:
        finite_profile_values = np.concatenate(finite_profile_parts)
        profile_ymin = np.min(finite_profile_values)
        profile_ymax = np.max(finite_profile_values)
        profile_span = profile_ymax - profile_ymin

        if profile_span == 0:
            profile_span = max(abs(profile_ymax), 1.0)

        profile_padding = (
            PROFILE_Y_PADDING_FRACTION * profile_span
        )

        ax_profile.set_ylim(
            profile_ymin - profile_padding,
            profile_ymax + profile_padding,
        )

    ax_profile.grid(
        True,
        alpha=0.3,
    )

    ax_profile.legend(
        loc="best",
    )

    plt.tight_layout()

    out_profile_png = (
        OUT_DIR
        / (
            f"BLO_keogram_profiles_"
            f"minus250_0_plus250km_"
            f"angle_{angle_deg:03.0f}.png"
        )
    )

    fig_profile.savefig(
        out_profile_png,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig_profile)

    saved_profile_plots.append(
        out_profile_png
    )

    # ----------------------------------------------
    # Save the three profiles as a NumPy file
    # ----------------------------------------------
    out_profile_npy = (
        OUT_DIR
        / (
            f"BLO_keogram_profiles_"
            f"minus250_0_plus250km_"
            f"angle_{angle_deg:03.0f}.npy"
        )
    )

    profile_array = np.column_stack(
        [
            times_min,
            profile_data[
                f"{distance_km[np.argmin(np.abs(distance_km + 250))]:.1f}_km"
            ],
            profile_data[
                f"{distance_km[np.argmin(np.abs(distance_km - 0))]:.1f}_km"
            ],
            profile_data[
                f"{distance_km[np.argmin(np.abs(distance_km - 250))]:.1f}_km"
            ],
        ]
    )

    np.save(
        out_profile_npy,
        profile_array,
    )


# ==================================================
# Save arrays and metadata
# ==================================================
np.save(
    OUT_DIR / "times_s.npy",
    times_s,
)

np.save(
    OUT_DIR / "times_min.npy",
    times_min,
)

np.save(
    OUT_DIR / "x_km.npy",
    x_km,
)

np.save(
    OUT_DIR / "y_km.npy",
    y_km,
)

np.save(
    OUT_DIR / "keogram_angles_deg.npy",
    KEOGRAM_ANGLES_DEG,
)

np.save(
    OUT_DIR / "first_frame_cut_angles_deg.npy",
    PLOT_CUT_ANGLES_DEG,
)

np.save(
    OUT_DIR / "profile_distances_km.npy",
    np.asarray(PROFILE_DISTANCES_KM),
)

with open(
    OUT_DIR / "Date_and_Start_Time.txt",
    "w",
) as f:
    f.write(f"Date: {date_start[0]}\n")
    f.write(f"Start time: {date_start[1]}\n")


# ==================================================
# Save final frame
# ==================================================
fig_last, ax_last = plt.subplots(figsize=(7, 6))

im_last = ax_last.imshow(
    wave_cube[-1],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        x_km.min(),
        x_km.max(),
        y_km.max(),
        y_km.min(),
    ],
)

cbar_last = fig_last.colorbar(
    im_last,
    ax=ax_last,
    fraction=0.046,
    pad=0.04,
)
cbar_last.set_label("Perturbation amplitude")

ax_last.set_title("Final BLO Perturbation Frame")
ax_last.set_xlabel("E-W distance (km)")
ax_last.set_ylabel("N-S distance (km)")
ax_last.set_aspect("equal")

plt.tight_layout()

out_last = OUT_DIR / "final_perturbation_frame.png"

fig_last.savefig(
    out_last,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_last)


# ==================================================
# Print outputs
# ==================================================
print("Saved:")
print(out_mp4)
print(out_first)
print(out_last)

print("\nDirectional keograms:")
for file_path in saved_keograms:
    print(file_path)

print("\nKeogram distance-profile plots:")
for file_path in saved_profile_plots:
    print(file_path)

print(
    f"\nNumber of directional keograms made: "
    f"{len(KEOGRAM_ANGLES_DEG)}"
)

print(
    f"Number of profile plots made: "
    f"{len(saved_profile_plots)}"
)

print(
    f"First-frame cut lines shown: every 15 degrees, "
    f"{len(PLOT_CUT_ANGLES_DEG)} lines"
)

print(
    f"Keogram/profile time range: 0 to "
    f"{KEOGRAM_TIME_MIN_MAX} min"
)

print(
    f"Keogram distance range: {KEOGRAM_DISTANCE_MIN} to "
    f"{KEOGRAM_DISTANCE_MAX} km"
)

print(
    "Each keogram and profile plot uses its own "
    "amplitude range."
)

print(
    "Profile distances requested:",
    PROFILE_DISTANCES_KM,
)

print("Output folder:", OUT_DIR)


Perturbation array shape: (180, 1000, 1000)
Date/start time: ['05_19_2026', '22:11:23']


C:\Users\domin\AppData\Local\Temp\ipykernel_3176\595559407.py:350: RuntimeWarning: Mean of empty slice
  keogram[it] = np.nanmean(


Saved:
BLO_perturbation_keogram_outputs_v3\May_19_2026\BLO_perturbation_animation.mp4
BLO_perturbation_keogram_outputs_v3\May_19_2026\first_perturbation_frame_with_keogram_cuts_every_15deg.png
BLO_perturbation_keogram_outputs_v3\May_19_2026\final_perturbation_frame.png

Directional keograms:
BLO_perturbation_keogram_outputs_v3\May_19_2026\BLO_keogram_angle_000.png
BLO_perturbation_keogram_outputs_v3\May_19_2026\BLO_keogram_angle_005.png
BLO_perturbation_keogram_outputs_v3\May_19_2026\BLO_keogram_angle_010.png
BLO_perturbation_keogram_outputs_v3\May_19_2026\BLO_keogram_angle_015.png
BLO_perturbation_keogram_outputs_v3\May_19_2026\BLO_keogram_angle_020.png
BLO_perturbation_keogram_outputs_v3\May_19_2026\BLO_keogram_angle_025.png
BLO_perturbation_keogram_outputs_v3\May_19_2026\BLO_keogram_angle_030.png
BLO_perturbation_keogram_outputs_v3\May_19_2026\BLO_keogram_angle_035.png
BLO_perturbation_keogram_outputs_v3\May_19_2026\BLO_keogram_angle_040.png
BLO_perturbation_keogram_outputs_v3\May_1

Date/start time: ['05_19_2026', '22:11:23']

### Calculate speed (old): 
650 km in 1500 s => 433 m/s

650 km is 25/120 of wavelength. => wavelength = 650 km /(25/120) = 3120 km. 

### New:
N-S direction:
250 km in 6.5x60 s = 390 s => 641 m/s
SE direction: 
641/sqrt2 = 453 m/s

## Also plot original keogram and smoothed ranged keogram. 

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.ndimage import gaussian_filter
from scipy.interpolate import PchipInterpolator
from scipy.signal import find_peaks


# ==================================================
# Settings
# ==================================================
BASE_OUT_DIR = Path(
    "BLO_perturbation_keogram_outputs_v3\May_19_2026"
)

# Process every directional keogram:
# 0, 5, 10, ..., 175 degrees
KEOGRAM_ANGLES_DEG = np.arange(
    0,
    176,
    5,
    dtype=int,
)

# Time range used for correlation
CORRELATION_TIME_MIN = 0.0
CORRELATION_TIME_MAX = 270.0

# Distance range included in the correlation map
CORRELATION_DISTANCE_MIN_KM = 0.0
CORRELATION_DISTANCE_MAX_KM = 250.0

# Shift range in minutes
SHIFT_MIN_MINUTES = -200.0
SHIFT_MAX_MINUTES = 200.0

# Minimum number of overlapping time samples
MIN_VALID_CORRELATION_POINTS = 20

# Minimum maximum-correlation value used in the slope fit
MIN_CORRELATION_FOR_SLOPE = 0.50

# Distance range used in the ridge fit
SLOPE_FIT_DISTANCE_MIN_KM = 0.0
SLOPE_FIT_DISTANCE_MAX_KM = 250.0

DPI = 200

CORRELATION_CMAP = "RdBu_r"
KEOGRAM_CMAP = "RdBu_r"

# Use symmetric automatic keogram limits based on this percentile.
# Set KEOGRAM_ABS_LIMIT to a number to force fixed limits.
KEOGRAM_ABS_LIMIT = None
KEOGRAM_AUTO_PERCENTILE = 99.0

# Correlation color scale
CORRELATION_VMIN = 0.70
CORRELATION_VMAX = 1.00

# Local maxima search on the 0 km correlation-shift profile.
# The left maximum is searched at negative shifts and the right
# maximum is searched at positive shifts. The zero-shift point is
# therefore excluded automatically.
LOCAL_MAXIMUM_MIN_PROMINENCE = 0.0
LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS = 1


# ==================================================
# Optional 2D smoothing
# ==================================================
APPLY_2D_SMOOTHING = True

# Keogram dimensions:
# axis 0 = time
# axis 1 = distance
SMOOTH_SIGMA_TIME_PIXELS = 1.5
SMOOTH_SIGMA_DISTANCE_PIXELS = 2.0

SMOOTH_MODE = "nearest"
SMOOTH_WEIGHT_TOLERANCE = 1e-12


# ==================================================
# Bias-correction settings
# ==================================================
APPLY_BIAS_CORRECTION = False

BIAS_TIME_POINTS_MIN = np.array(
    [
        0.0,
        150.0,
        300.0,
    ],
    dtype=float,
)

BIAS_AT_MAX_DISTANCE = np.array(
    [
        0.20,
        0.04,
        -0.01,
    ],
    dtype=float,
)

BIAS_DISTANCE_MIN_KM = 0.0
BIAS_DISTANCE_MAX_KM = 250.0

# 1.0 = linear distance dependence
# 2.0 = quadratic distance dependence
BIAS_DISTANCE_POWER = 1.0

BIAS_ZERO_OUTSIDE_DISTANCE_RANGE = True
BIAS_ZERO_OUTSIDE_TIME_RANGE = True
CLIP_NEGATIVE_TIME_CORRECTION = False


# ==================================================
# Filename-safe numeric label
# ==================================================
def safe_number_label(value):
    value_float = float(value)

    if value_float.is_integer():
        return str(
            int(value_float)
        )

    return (
        f"{value_float:g}"
        .replace("-", "m")
        .replace(".", "p")
    )


# ==================================================
# NaN-aware 2D Gaussian smoothing
# ==================================================
def smooth_keogram_2d(
    data,
    sigma_time,
    sigma_distance,
    mode="nearest",
    weight_tolerance=1e-12,
):
    """
    Apply NaN-aware Gaussian smoothing to a 2D keogram.
    """
    data = np.asarray(
        data,
        dtype=float,
    )

    if data.ndim != 2:
        raise ValueError(
            "The keogram must be two-dimensional."
        )

    if sigma_time < 0 or sigma_distance < 0:
        raise ValueError(
            "Smoothing sigma values must be nonnegative."
        )

    if sigma_time == 0 and sigma_distance == 0:
        return data.copy()

    valid_mask = np.isfinite(
        data
    )

    data_filled = np.where(
        valid_mask,
        data,
        0.0,
    )

    weights = valid_mask.astype(
        float
    )

    sigma = (
        sigma_time,
        sigma_distance,
    )

    smoothed_values = gaussian_filter(
        data_filled,
        sigma=sigma,
        mode=mode,
    )

    smoothed_weights = gaussian_filter(
        weights,
        sigma=sigma,
        mode=mode,
    )

    smoothed = np.full(
        data.shape,
        np.nan,
        dtype=float,
    )

    valid_output = (
        smoothed_weights
        > weight_tolerance
    )

    smoothed[valid_output] = (
        smoothed_values[valid_output]
        / smoothed_weights[valid_output]
    )

    return smoothed


# ==================================================
# Create 2D bias-correction keogram
# ==================================================
def create_bias_correction_keogram(
    times_min,
    distance_km,
    bias_time_points_min,
    bias_at_max_distance,
    bias_distance_min_km,
    bias_distance_max_km,
    distance_power=1.0,
    zero_outside_distance_range=True,
    zero_outside_time_range=True,
    clip_negative_time_correction=False,
):
    """
    Create the time x distance additive bias field.
    """
    times_min = np.asarray(
        times_min,
        dtype=float,
    )

    distance_km = np.asarray(
        distance_km,
        dtype=float,
    )

    bias_time_points_min = np.asarray(
        bias_time_points_min,
        dtype=float,
    )

    bias_at_max_distance = np.asarray(
        bias_at_max_distance,
        dtype=float,
    )

    if (
        len(bias_time_points_min)
        != len(bias_at_max_distance)
    ):
        raise ValueError(
            "Bias time points and bias values must have "
            "the same length."
        )

    if not np.all(
        np.diff(bias_time_points_min) > 0
    ):
        raise ValueError(
            "BIAS_TIME_POINTS_MIN must be strictly increasing."
        )

    if bias_distance_max_km <= bias_distance_min_km:
        raise ValueError(
            "The maximum bias distance must exceed "
            "the minimum bias distance."
        )

    if distance_power <= 0:
        raise ValueError(
            "BIAS_DISTANCE_POWER must be positive."
        )

    # ==================================================
    # PCHIP time dependence
    # ==================================================
    interpolator = PchipInterpolator(
        bias_time_points_min,
        bias_at_max_distance,
        extrapolate=False,
    )

    bias_vs_time = interpolator(
        times_min
    )

    if zero_outside_time_range:
        inside_time = (
            (times_min >= bias_time_points_min[0])
            & (times_min <= bias_time_points_min[-1])
        )

        bias_vs_time = np.where(
            inside_time,
            bias_vs_time,
            0.0,
        )
    else:
        bias_vs_time = np.where(
            times_min < bias_time_points_min[0],
            bias_at_max_distance[0],
            bias_vs_time,
        )

        bias_vs_time = np.where(
            times_min > bias_time_points_min[-1],
            bias_at_max_distance[-1],
            bias_vs_time,
        )

    if clip_negative_time_correction:
        bias_vs_time = np.maximum(
            bias_vs_time,
            0.0,
        )

    # ==================================================
    # Distance dependence
    # ==================================================
    normalized_distance = (
        distance_km
        - bias_distance_min_km
    ) / (
        bias_distance_max_km
        - bias_distance_min_km
    )

    if zero_outside_distance_range:
        inside_distance = (
            (distance_km >= bias_distance_min_km)
            & (distance_km <= bias_distance_max_km)
        )

        normalized_distance = np.where(
            inside_distance,
            normalized_distance,
            0.0,
        )
    else:
        normalized_distance = np.clip(
            normalized_distance,
            0.0,
            1.0,
        )

    distance_factor = (
        normalized_distance ** distance_power
    )

    correction_keogram = (
        bias_vs_time[:, np.newaxis]
        * distance_factor[np.newaxis, :]
    )

    return (
        correction_keogram,
        bias_vs_time,
        distance_factor,
    )


# ==================================================
# NaN-aware Pearson correlation
# ==================================================
def nan_pearson_correlation(
    reference,
    comparison,
    minimum_points=20,
):
    """
    Calculate Pearson correlation using finite overlapping values.
    """
    reference = np.asarray(
        reference,
        dtype=float,
    )

    comparison = np.asarray(
        comparison,
        dtype=float,
    )

    if reference.shape != comparison.shape:
        raise ValueError(
            "The two arrays must have the same shape."
        )

    valid = (
        np.isfinite(reference)
        & np.isfinite(comparison)
    )

    if np.count_nonzero(valid) < minimum_points:
        return np.nan

    reference_valid = reference[
        valid
    ]

    comparison_valid = comparison[
        valid
    ]

    reference_centered = (
        reference_valid
        - np.mean(reference_valid)
    )

    comparison_centered = (
        comparison_valid
        - np.mean(comparison_valid)
    )

    denominator = np.sqrt(
        np.sum(reference_centered ** 2)
        * np.sum(comparison_centered ** 2)
    )

    if (
        not np.isfinite(denominator)
        or denominator <= 0
    ):
        return np.nan

    correlation = (
        np.sum(
            reference_centered
            * comparison_centered
        )
        / denominator
    )

    return np.clip(
        correlation,
        -1.0,
        1.0,
    )


# ==================================================
# Correlation for one integer-pixel shift
# ==================================================
def calculate_shifted_correlation(
    reference_row,
    comparison_row,
    shift_pixels,
    minimum_points=20,
):
    """
    Shift comparison_row and correlate it with reference_row.

    Positive shift:
        comparison row moves toward later times.

    Negative shift:
        comparison row moves toward earlier times.

    No circular wrapping is used.
    """
    reference_row = np.asarray(
        reference_row,
        dtype=float,
    )

    comparison_row = np.asarray(
        comparison_row,
        dtype=float,
    )

    if reference_row.shape != comparison_row.shape:
        raise ValueError(
            "Reference and comparison rows must have "
            "the same shape."
        )

    ntime = len(
        reference_row
    )

    if abs(shift_pixels) >= ntime:
        return np.nan

    if shift_pixels > 0:
        reference_overlap = reference_row[
            shift_pixels:
        ]

        comparison_overlap = comparison_row[
            :-shift_pixels
        ]

    elif shift_pixels < 0:
        shift_abs = abs(
            shift_pixels
        )

        reference_overlap = reference_row[
            :-shift_abs
        ]

        comparison_overlap = comparison_row[
            shift_abs:
        ]

    else:
        reference_overlap = reference_row
        comparison_overlap = comparison_row

    return nan_pearson_correlation(
        reference=reference_overlap,
        comparison=comparison_overlap,
        minimum_points=minimum_points,
    )


# ==================================================
# Calculate complete correlation map
# ==================================================
def calculate_correlation_map(
    keogram,
    distance_km,
    reference_distance_km,
    selected_distance_mask,
    shifts_pixels,
    minimum_points=20,
):
    """
    Correlate every selected distance row with the reference row
    for every requested shift.
    """
    keogram = np.asarray(
        keogram,
        dtype=float,
    )

    distance_km = np.asarray(
        distance_km,
        dtype=float,
    )

    shifts_pixels = np.asarray(
        shifts_pixels,
        dtype=int,
    )

    reference_index = np.argmin(
        np.abs(
            distance_km
            - reference_distance_km
        )
    )

    selected_indices = np.where(
        selected_distance_mask
    )[0]

    selected_distances = distance_km[
        selected_indices
    ]

    reference_row = keogram[
        :,
        reference_index,
    ]

    correlation_map = np.full(
        (
            len(selected_indices),
            len(shifts_pixels),
        ),
        np.nan,
        dtype=float,
    )

    for output_row_index, keogram_row_index in enumerate(
        selected_indices
    ):
        comparison_row = keogram[
            :,
            keogram_row_index,
        ]

        for shift_index, shift_pixels in enumerate(
            shifts_pixels
        ):
            correlation_map[
                output_row_index,
                shift_index,
            ] = calculate_shifted_correlation(
                reference_row=reference_row,
                comparison_row=comparison_row,
                shift_pixels=int(shift_pixels),
                minimum_points=minimum_points,
            )

    return (
        correlation_map,
        selected_distances,
        reference_index,
    )


# ==================================================
# Maximum-correlation shift for each row
# ==================================================
def find_best_shifts(
    correlation_map,
    shifts_minutes,
):
    """
    Find the shift in minutes having the largest correlation
    for every distance row.
    """
    correlation_map = np.asarray(
        correlation_map,
        dtype=float,
    )

    shifts_minutes = np.asarray(
        shifts_minutes,
        dtype=float,
    )

    best_shift_minutes = np.full(
        correlation_map.shape[0],
        np.nan,
        dtype=float,
    )

    maximum_correlation = np.full(
        correlation_map.shape[0],
        np.nan,
        dtype=float,
    )

    for row_index in range(
        correlation_map.shape[0]
    ):
        correlation_row = correlation_map[
            row_index,
            :
        ]

        finite = np.isfinite(
            correlation_row
        )

        if not np.any(finite):
            continue

        finite_indices = np.where(
            finite
        )[0]

        local_max_index = np.argmax(
            correlation_row[finite]
        )

        full_max_index = finite_indices[
            local_max_index
        ]

        best_shift_minutes[row_index] = (
            shifts_minutes[full_max_index]
        )

        maximum_correlation[row_index] = (
            correlation_row[full_max_index]
        )

    return (
        best_shift_minutes,
        maximum_correlation,
    )


# ==================================================
# Extract the correlation-shift profile at 0 km
# ==================================================
def extract_zero_km_correlation_profile(
    correlation_map,
    selected_distances,
):
    """
    Return the correlation-versus-shift profile for the selected
    distance row nearest 0 km.
    """
    correlation_map = np.asarray(
        correlation_map,
        dtype=float,
    )

    selected_distances = np.asarray(
        selected_distances,
        dtype=float,
    )

    if correlation_map.ndim != 2:
        raise ValueError(
            "correlation_map must be two-dimensional."
        )

    if correlation_map.shape[0] != len(selected_distances):
        raise ValueError(
            "The number of correlation-map rows must equal "
            "the number of selected distances."
        )

    zero_distance_row_index = int(
        np.argmin(
            np.abs(selected_distances)
        )
    )

    zero_distance_km = float(
        selected_distances[zero_distance_row_index]
    )

    zero_km_profile = correlation_map[
        zero_distance_row_index,
        :
    ].copy()

    return (
        zero_km_profile,
        zero_distance_row_index,
        zero_distance_km,
    )


# ==================================================
# Find strongest local maximum on each side of zero
# ==================================================
def find_left_and_right_local_maxima(
    correlation_profile,
    shifts_minutes,
    minimum_prominence=0.0,
    minimum_distance_pixels=1,
):
    """
    Find the strongest finite local maximum at negative shift
    and the strongest finite local maximum at positive shift.

    The point at zero shift is not included in either search.

    If scipy.signal.find_peaks finds no strict local maximum on one
    side, the largest finite value on that side is used as a fallback.
    """
    correlation_profile = np.asarray(
        correlation_profile,
        dtype=float,
    )

    shifts_minutes = np.asarray(
        shifts_minutes,
        dtype=float,
    )

    if correlation_profile.shape != shifts_minutes.shape:
        raise ValueError(
            "correlation_profile and shifts_minutes must "
            "have the same shape."
        )

    def find_one_side(side_mask):
        side_indices = np.where(
            side_mask
            & np.isfinite(correlation_profile)
        )[0]

        if len(side_indices) == 0:
            return {
                "index": -1,
                "shift_minutes": np.nan,
                "correlation": np.nan,
                "used_fallback": False,
            }

        side_values = correlation_profile[
            side_indices
        ]

        local_indices, properties = find_peaks(
            side_values,
            prominence=minimum_prominence,
            distance=minimum_distance_pixels,
        )

        used_fallback = False

        if len(local_indices) > 0:
            local_values = side_values[
                local_indices
            ]

            strongest_local_position = local_indices[
                int(np.nanargmax(local_values))
            ]

            full_index = int(
                side_indices[strongest_local_position]
            )
        else:
            used_fallback = True

            strongest_side_position = int(
                np.nanargmax(side_values)
            )

            full_index = int(
                side_indices[strongest_side_position]
            )

        return {
            "index": full_index,
            "shift_minutes": float(
                shifts_minutes[full_index]
            ),
            "correlation": float(
                correlation_profile[full_index]
            ),
            "used_fallback": used_fallback,
        }

    left_result = find_one_side(
        shifts_minutes < 0.0
    )

    right_result = find_one_side(
        shifts_minutes > 0.0
    )

    return {
        "left": left_result,
        "right": right_result,
    }


# ==================================================
# Fit maximum-correlation ridge
# ==================================================
def fit_maximum_correlation_slope(
    distances_km,
    best_shift_minutes,
    maximum_correlation,
    minimum_correlation=0.5,
    distance_min_km=None,
    distance_max_km=None,
):
    """
    Fit:

        shift_minutes = slope * distance_km + intercept
    """
    distances_km = np.asarray(
        distances_km,
        dtype=float,
    )

    best_shift_minutes = np.asarray(
        best_shift_minutes,
        dtype=float,
    )

    maximum_correlation = np.asarray(
        maximum_correlation,
        dtype=float,
    )

    fit_mask = (
        np.isfinite(distances_km)
        & np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (
            maximum_correlation
            >= minimum_correlation
        )
    )

    if distance_min_km is not None:
        fit_mask &= (
            distances_km
            >= distance_min_km
        )

    if distance_max_km is not None:
        fit_mask &= (
            distances_km
            <= distance_max_km
        )

    number_of_points = np.count_nonzero(
        fit_mask
    )

    if number_of_points < 2:
        return {
            "slope_min_per_km": np.nan,
            "intercept_minutes": np.nan,
            "speed_km_per_min": np.nan,
            "speed_m_per_s": np.nan,
            "r_squared": np.nan,
            "number_of_fit_points": number_of_points,
            "fit_mask": fit_mask,
        }

    fit_distances = distances_km[
        fit_mask
    ]

    fit_shifts = best_shift_minutes[
        fit_mask
    ]

    slope_min_per_km, intercept_minutes = np.polyfit(
        fit_distances,
        fit_shifts,
        deg=1,
    )

    predicted_shifts = (
        slope_min_per_km
        * fit_distances
        + intercept_minutes
    )

    residual_sum_squares = np.sum(
        (
            fit_shifts
            - predicted_shifts
        ) ** 2
    )

    total_sum_squares = np.sum(
        (
            fit_shifts
            - np.mean(fit_shifts)
        ) ** 2
    )

    if total_sum_squares > 0:
        r_squared = (
            1.0
            - residual_sum_squares
            / total_sum_squares
        )
    else:
        r_squared = np.nan

    if np.isclose(
        slope_min_per_km,
        0.0,
    ):
        speed_km_per_min = np.nan
        speed_m_per_s = np.nan
    else:
        speed_km_per_min = (
            1.0
            / slope_min_per_km
        )

        speed_m_per_s = (
            speed_km_per_min
            * 1000.0
            / 60.0
        )

    return {
        "slope_min_per_km": slope_min_per_km,
        "intercept_minutes": intercept_minutes,
        "speed_km_per_min": speed_km_per_min,
        "speed_m_per_s": speed_m_per_s,
        "r_squared": r_squared,
        "number_of_fit_points": number_of_points,
        "fit_mask": fit_mask,
    }


# ==================================================
# Keogram color limit
# ==================================================
def determine_keogram_abs_limit(
    full_keogram,
    limited_keogram,
):
    """
    Return one symmetric absolute color limit shared by both
    keogram panels.
    """
    if KEOGRAM_ABS_LIMIT is not None:
        abs_limit = float(
            KEOGRAM_ABS_LIMIT
        )

        if (
            not np.isfinite(abs_limit)
            or abs_limit <= 0
        ):
            raise ValueError(
                "KEOGRAM_ABS_LIMIT must be positive and finite."
            )

        return abs_limit

    finite_values = []

    for data in [
        full_keogram,
        limited_keogram,
    ]:
        data = np.asarray(
            data,
            dtype=float,
        )

        finite = np.abs(
            data[np.isfinite(data)]
        )

        if finite.size > 0:
            finite_values.append(
                finite
            )

    if len(finite_values) == 0:
        return 1.0

    all_finite = np.concatenate(
        finite_values
    )

    abs_limit = np.nanpercentile(
        all_finite,
        KEOGRAM_AUTO_PERCENTILE,
    )

    if (
        not np.isfinite(abs_limit)
        or abs_limit <= 0
    ):
        abs_limit = np.nanmax(
            all_finite
        )

    if (
        not np.isfinite(abs_limit)
        or abs_limit <= 0
    ):
        abs_limit = 1.0

    return float(
        abs_limit
    )


# ==================================================
# Plot keograms, correlation map, ridge, and profile
# ==================================================
def plot_correlation_map(
    angle_deg,
    full_keogram,
    full_times_min,
    full_distance_km,
    limited_keogram,
    limited_times_min,
    limited_distance_km,
    correlation_map,
    shifts_minutes,
    selected_distances,
    best_shift_minutes,
    maximum_correlation,
    slope_results,
    zero_km_correlation_profile,
    zero_km_distance,
    zero_km_local_maxima,
    dataset_name,
    output_file,
):
    """
    Plot:
      1. full-range original keogram;
      2. limited-range smoothed/corrected keogram;
      3. correlation versus time shift and distance;
      4. correlation versus shift for the 0 km row.
    """
    fig, (
        ax_full,
        ax_limited,
        ax_map,
        ax_profile,
    ) = plt.subplots(
        nrows=4,
        ncols=1,
        figsize=(11, 16),
        gridspec_kw={
            "height_ratios": [
                1.55,
                1.55,
                2.5,
                1.15,
            ],
        },
    )

    keogram_abs_limit = determine_keogram_abs_limit(
        full_keogram=full_keogram,
        limited_keogram=limited_keogram,
    )

    # ==================================================
    # Full-range original keogram
    # ==================================================
    full_mesh = ax_full.pcolormesh(
        full_times_min,
        full_distance_km,
        full_keogram.T,
        shading="auto",
        cmap=KEOGRAM_CMAP,
        vmin=-keogram_abs_limit,
        vmax=keogram_abs_limit,
    )

    full_cbar = fig.colorbar(
        full_mesh,
        ax=ax_full,
        fraction=0.025,
        pad=0.02,
    )

    full_cbar.set_label(
        "Keogram value"
    )

    ax_full.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
    )

    ax_full.set_xlim(
        np.nanmin(full_times_min),
        np.nanmax(full_times_min),
    )

    ax_full.set_ylim(
        np.nanmin(full_distance_km),
        np.nanmax(full_distance_km),
    )

    ax_full.set_title(
        f"{angle_deg}° Full-Range Original Keogram"
    )

    ax_full.set_xlabel(
        "Time (min)"
    )

    ax_full.set_ylabel(
        "Distance along slice (km)"
    )

    # ==================================================
    # Limited-range processed keogram
    # ==================================================
    limited_mesh = ax_limited.pcolormesh(
        limited_times_min,
        limited_distance_km,
        limited_keogram.T,
        shading="auto",
        cmap=KEOGRAM_CMAP,
        vmin=-keogram_abs_limit,
        vmax=keogram_abs_limit,
    )

    limited_cbar = fig.colorbar(
        limited_mesh,
        ax=ax_limited,
        fraction=0.025,
        pad=0.02,
    )

    limited_cbar.set_label(
        "Keogram value"
    )

    ax_limited.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
    )

    ax_limited.set_xlim(
        CORRELATION_TIME_MIN,
        CORRELATION_TIME_MAX,
    )

    ax_limited.set_ylim(
        CORRELATION_DISTANCE_MIN_KM,
        CORRELATION_DISTANCE_MAX_KM,
    )

    ax_limited.set_title(
        "Limited-Range Processed Keogram\n"
        f"{dataset_name}"
    )

    ax_limited.set_xlabel(
        "Time (min)"
    )

    ax_limited.set_ylabel(
        "Distance along slice (km)"
    )

    # ==================================================
    # Correlation map
    # ==================================================
    if len(shifts_minutes) > 1:
        shift_spacing = np.median(
            np.diff(shifts_minutes)
        )
    else:
        shift_spacing = 1.0

    if len(selected_distances) > 1:
        distance_spacing = np.median(
            np.diff(selected_distances)
        )
    else:
        distance_spacing = 1.0

    im = ax_map.imshow(
        correlation_map,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap=CORRELATION_CMAP,
        vmin=CORRELATION_VMIN,
        vmax=CORRELATION_VMAX,
        extent=[
            shifts_minutes[0]
            - shift_spacing / 2.0,
            shifts_minutes[-1]
            + shift_spacing / 2.0,
            selected_distances[0]
            - distance_spacing / 2.0,
            selected_distances[-1]
            + distance_spacing / 2.0,
        ],
    )

    cbar = fig.colorbar(
        im,
        ax=ax_map,
        fraction=0.025,
        pad=0.02,
    )

    cbar.set_label(
        "Pearson correlation with 0 km row"
    )

    ax_map.axvline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.2,
        label="Zero shift",
    )

    ax_map.axhline(
        zero_km_distance,
        color="white",
        linestyle=":",
        linewidth=1.5,
        label=f"Profile row ({zero_km_distance:g} km)",
    )

    ridge_mask = (
        np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (
            maximum_correlation
            >= MIN_CORRELATION_FOR_SLOPE
        )
    )

    if np.any(ridge_mask):
        ax_map.plot(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            color="black",
            linewidth=1.2,
            label="Maximum correlation",
        )

        ax_map.scatter(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            color="black",
            s=8,
        )

    slope = slope_results[
        "slope_min_per_km"
    ]

    intercept = slope_results[
        "intercept_minutes"
    ]

    r_squared = slope_results[
        "r_squared"
    ]

    speed_m_per_s = slope_results[
        "speed_m_per_s"
    ]

    if np.isfinite(slope):
        fit_distances = np.linspace(
            SLOPE_FIT_DISTANCE_MIN_KM,
            SLOPE_FIT_DISTANCE_MAX_KM,
            300,
        )

        fit_shifts = (
            slope
            * fit_distances
            + intercept
        )

        ax_map.plot(
            fit_shifts,
            fit_distances,
            color="lime",
            linestyle="--",
            linewidth=2.0,
            label="Linear ridge fit",
        )

        fit_text = (
            f"Shift slope = {slope:.5f} min/km\n"
            f"Equivalent speed = {speed_m_per_s:.2f} m/s\n"
            f"$R^2$ = {r_squared:.3f}"
        )

        ax_map.text(
            0.02,
            0.03,
            fit_text,
            transform=ax_map.transAxes,
            ha="left",
            va="bottom",
            fontsize=9,
            bbox={
                "facecolor": "white",
                "edgecolor": "black",
                "alpha": 0.85,
            },
        )

    ax_map.set_xlim(
        SHIFT_MIN_MINUTES,
        SHIFT_MAX_MINUTES,
    )

    ax_map.set_ylim(
        CORRELATION_DISTANCE_MIN_KM,
        CORRELATION_DISTANCE_MAX_KM,
    )

    ax_map.set_title(
        f"{angle_deg}° Shifted Row Correlation\n"
        f"{dataset_name}"
    )

    ax_map.set_xlabel(
        "Shift applied to row (min)"
    )

    ax_map.set_ylabel(
        "Distance along slice (km)"
    )

    ax_map.legend(
        loc="upper right",
        fontsize=8,
    )

    # ==================================================
    # 0 km correlation-shift line
    # ==================================================
    ax_profile.plot(
        shifts_minutes,
        zero_km_correlation_profile,
        color="black",
        linewidth=1.4,
        label=f"{zero_km_distance:g} km correlation profile",
    )

    ax_profile.axvline(
        0.0,
        color="gray",
        linestyle="--",
        linewidth=1.0,
    )

    left_peak = zero_km_local_maxima[
        "left"
    ]

    right_peak = zero_km_local_maxima[
        "right"
    ]

    if np.isfinite(
        left_peak["shift_minutes"]
    ):
        ax_profile.scatter(
            left_peak["shift_minutes"],
            left_peak["correlation"],
            s=55,
            marker="o",
            zorder=5,
            label=(
                "Left local max: "
                f"{left_peak['shift_minutes']:.2f} min"
            ),
        )

        ax_profile.axvline(
            left_peak["shift_minutes"],
            linestyle=":",
            linewidth=1.2,
        )

        ax_profile.annotate(
            (
                f"{left_peak['shift_minutes']:.2f} min\n"
                f"r={left_peak['correlation']:.3f}"
            ),
            xy=(
                left_peak["shift_minutes"],
                left_peak["correlation"],
            ),
            xytext=(8, 15),
            textcoords="offset points",
            ha="left",
            va="bottom",
            fontsize=8,
            arrowprops={
                "arrowstyle": "->",
                "linewidth": 0.8,
            },
        )

    if np.isfinite(
        right_peak["shift_minutes"]
    ):
        ax_profile.scatter(
            right_peak["shift_minutes"],
            right_peak["correlation"],
            s=55,
            marker="s",
            zorder=5,
            label=(
                "Right local max: "
                f"{right_peak['shift_minutes']:.2f} min"
            ),
        )

        ax_profile.axvline(
            right_peak["shift_minutes"],
            linestyle=":",
            linewidth=1.2,
        )

        ax_profile.annotate(
            (
                f"{right_peak['shift_minutes']:.2f} min\n"
                f"r={right_peak['correlation']:.3f}"
            ),
            xy=(
                right_peak["shift_minutes"],
                right_peak["correlation"],
            ),
            xytext=(-8, 15),
            textcoords="offset points",
            ha="right",
            va="bottom",
            fontsize=8,
            arrowprops={
                "arrowstyle": "->",
                "linewidth": 0.8,
            },
        )

    ax_profile.set_xlim(
        SHIFT_MIN_MINUTES,
        SHIFT_MAX_MINUTES,
    )

    ax_profile.set_xlabel(
        "Shift applied to row (min)"
    )

    ax_profile.set_ylabel(
        "Correlation"
    )

    ax_profile.set_title(
        f"Correlation-Shift Line at {zero_km_distance:g} km"
    )

    ax_profile.grid(
        True,
        alpha=0.3,
    )

    ax_profile.legend(
        loc="best",
        fontsize=8,
    )

    plt.tight_layout()

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig)


# ==================================================
# Print slope results
# ==================================================
def print_slope_results(
    angle_deg,
    label,
    results,
):
    print(
        f"\n{angle_deg}° — {label}:"
    )

    print(
        f"  Shift slope: "
        f"{results['slope_min_per_km']:.8f} min/km"
    )

    print(
        f"  Intercept: "
        f"{results['intercept_minutes']:.6f} min"
    )

    print(
        f"  Equivalent speed: "
        f"{results['speed_m_per_s']:.4f} m/s"
    )

    print(
        f"  R²: "
        f"{results['r_squared']:.6f}"
    )

    print(
        f"  Points used: "
        f"{results['number_of_fit_points']}"
    )


# ==================================================
# Output paths
# ==================================================
sigma_t_label = safe_number_label(
    SMOOTH_SIGMA_TIME_PIXELS
)

sigma_d_label = safe_number_label(
    SMOOTH_SIGMA_DISTANCE_PIXELS
)

distance_folder = (
    f"r_"
    f"{int(CORRELATION_DISTANCE_MIN_KM)}_"
    f"{int(CORRELATION_DISTANCE_MAX_KM)}"
)

shift_folder = (
    f"shift_min_"
    f"{safe_number_label(SHIFT_MIN_MINUTES)}_"
    f"{safe_number_label(SHIFT_MAX_MINUTES)}"
)

if APPLY_2D_SMOOTHING:
    smoothing_folder = (
        f"sig_"
        f"{sigma_t_label}_"
        f"{sigma_d_label}"
    )
else:
    smoothing_folder = "raw"

OUT_DIR = (
    BASE_OUT_DIR
    / distance_folder
    / "row_correlation_minutes_all_angles"
    / shift_folder
)

UNCORRECTED_OUT_DIR = (
    OUT_DIR
    / "uncorrected"
)

CORRECTED_OUT_DIR = (
    OUT_DIR
    / "corrected"
)

SLOPE_REPORT_OUT_DIR = (
    OUT_DIR
    / "slope_reports"
)

for directory in [
    OUT_DIR,
    UNCORRECTED_OUT_DIR,
    CORRECTED_OUT_DIR,
    SLOPE_REPORT_OUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==================================================
# Read common time array
# ==================================================
TIMES_FILE = (
    BASE_OUT_DIR
    / "times_s.npy"
)

times_s = np.load(
    TIMES_FILE
).astype(float)

times_min = times_s / 60.0

if len(times_min) < 2:
    raise ValueError(
        "At least two time samples are required."
    )


# ==================================================
# Determine minutes per time pixel
# ==================================================
time_differences_min = np.diff(
    times_min
)

finite_time_differences = time_differences_min[
    np.isfinite(time_differences_min)
]

if len(finite_time_differences) == 0:
    raise ValueError(
        "Could not determine the time spacing."
    )

MINUTES_PER_PIXEL = np.median(
    finite_time_differences
)

if (
    not np.isfinite(MINUTES_PER_PIXEL)
    or MINUTES_PER_PIXEL <= 0
):
    raise ValueError(
        "The calculated minutes-per-pixel value is invalid."
    )

maximum_time_spacing_deviation = np.max(
    np.abs(
        finite_time_differences
        - MINUTES_PER_PIXEL
    )
)

print(
    "Angles to process:"
)

print(
    KEOGRAM_ANGLES_DEG
)

print(
    f"Number of angles: "
    f"{len(KEOGRAM_ANGLES_DEG)}"
)

print(
    "\nTime sampling:"
)

print(
    f"One time pixel = "
    f"{MINUTES_PER_PIXEL:.6f} minutes"
)

print(
    f"One time pixel = "
    f"{MINUTES_PER_PIXEL * 60.0:.3f} seconds"
)

print(
    f"Maximum spacing deviation = "
    f"{maximum_time_spacing_deviation:.6e} minutes"
)


# ==================================================
# Convert minute shifts to integer pixel shifts
# ==================================================
shift_min_pixels = int(
    np.ceil(
        SHIFT_MIN_MINUTES
        / MINUTES_PER_PIXEL
    )
)

shift_max_pixels = int(
    np.floor(
        SHIFT_MAX_MINUTES
        / MINUTES_PER_PIXEL
    )
)

shifts_pixels = np.arange(
    shift_min_pixels,
    shift_max_pixels + 1,
    dtype=int,
)

shifts_minutes = (
    shifts_pixels
    * MINUTES_PER_PIXEL
)

print(
    "\nShift range:"
)

print(
    f"Requested: "
    f"{SHIFT_MIN_MINUTES:.2f} to "
    f"{SHIFT_MAX_MINUTES:.2f} min"
)

print(
    f"Actual: "
    f"{shifts_minutes[0]:.2f} to "
    f"{shifts_minutes[-1]:.2f} min"
)

print(
    f"Pixel shifts: "
    f"{shifts_pixels[0]} to "
    f"{shifts_pixels[-1]}"
)


# ==================================================
# Select common time range
# ==================================================
time_mask = (
    (times_min >= CORRELATION_TIME_MIN)
    & (times_min <= CORRELATION_TIME_MAX)
)

if np.count_nonzero(time_mask) < MIN_VALID_CORRELATION_POINTS:
    raise ValueError(
        "The selected time interval has too few samples."
    )

times_correlation_min = times_min[
    time_mask
]


# ==================================================
# Save common arrays
# ==================================================
np.save(
    OUT_DIR / "angles_deg.npy",
    KEOGRAM_ANGLES_DEG,
)

np.save(
    OUT_DIR / "shifts_minutes.npy",
    shifts_minutes,
)

np.save(
    OUT_DIR / "shifts_pixels.npy",
    shifts_pixels,
)

np.save(
    OUT_DIR / "correlation_times_min.npy",
    times_correlation_min,
)


# ==================================================
# Storage for combined angle results
# ==================================================
combined_results = []
combined_zero_km_peak_results = []

successful_angles = []
failed_angles = []


# ==================================================
# Process every angle
# ==================================================
for angle_number, angle_deg in enumerate(
    KEOGRAM_ANGLES_DEG,
    start=1,
):
    print(
        "\n"
        + "=" * 70
    )

    print(
        f"Processing angle "
        f"{angle_deg}° "
        f"({angle_number}/{len(KEOGRAM_ANGLES_DEG)})"
    )

    print(
        "=" * 70
    )

    KEOGRAM_FILE = (
        BASE_OUT_DIR
        / f"BLO_keogram_angle_{angle_deg:03d}.npy"
    )

    if not KEOGRAM_FILE.exists():
        print(
            f"WARNING: File not found: "
            f"{KEOGRAM_FILE}"
        )

        failed_angles.append(
            (
                int(angle_deg),
                "file not found",
            )
        )

        continue

    try:
        # ==================================================
        # Read keogram
        # ==================================================
        keogram_original = np.load(
            KEOGRAM_FILE
        ).astype(float)

        if keogram_original.ndim != 2:
            raise ValueError(
                f"Expected a 2D keogram, found "
                f"{keogram_original.shape}."
            )

        nt, nspace = keogram_original.shape

        if len(times_min) != nt:
            raise ValueError(
                f"Keogram has {nt} time samples, but "
                f"times_s.npy has {len(times_min)}."
            )

        # ==================================================
        # Construct distance axis
        # ==================================================
        distance_km = np.arange(
            -(nspace // 2),
            nspace // 2 + 1,
            dtype=float,
        )

        if len(distance_km) != nspace:
            distance_km = (
                np.arange(
                    nspace,
                    dtype=float,
                )
                - nspace // 2
            )

        available_distance_min = np.nanmin(
            distance_km
        )

        available_distance_max = np.nanmax(
            distance_km
        )

        if (
            CORRELATION_DISTANCE_MIN_KM
            < available_distance_min
            or CORRELATION_DISTANCE_MAX_KM
            > available_distance_max
        ):
            raise ValueError(
                f"Requested distance range "
                f"{CORRELATION_DISTANCE_MIN_KM} to "
                f"{CORRELATION_DISTANCE_MAX_KM} km "
                f"is outside available range "
                f"{available_distance_min} to "
                f"{available_distance_max} km."
            )

        # ==================================================
        # Select time interval
        # ==================================================
        keogram_time_selected = keogram_original[
            time_mask,
            :
        ]

        # ==================================================
        # Optional smoothing
        # ==================================================
        if APPLY_2D_SMOOTHING:
            keogram_uncorrected = smooth_keogram_2d(
                data=keogram_time_selected,
                sigma_time=SMOOTH_SIGMA_TIME_PIXELS,
                sigma_distance=SMOOTH_SIGMA_DISTANCE_PIXELS,
                mode=SMOOTH_MODE,
                weight_tolerance=SMOOTH_WEIGHT_TOLERANCE,
            )

            uncorrected_dataset_name = (
                f"Uncorrected, smoothed "
                f"σt={SMOOTH_SIGMA_TIME_PIXELS:g}, "
                f"σd={SMOOTH_SIGMA_DISTANCE_PIXELS:g}"
            )
        else:
            keogram_uncorrected = (
                keogram_time_selected.copy()
            )

            uncorrected_dataset_name = (
                "Uncorrected, unsmoothed"
            )

        # ==================================================
        # Create bias correction
        # ==================================================
        (
            bias_correction_keogram,
            bias_vs_time,
            bias_distance_factor,
        ) = create_bias_correction_keogram(
            times_min=times_correlation_min,
            distance_km=distance_km,
            bias_time_points_min=BIAS_TIME_POINTS_MIN,
            bias_at_max_distance=BIAS_AT_MAX_DISTANCE,
            bias_distance_min_km=BIAS_DISTANCE_MIN_KM,
            bias_distance_max_km=BIAS_DISTANCE_MAX_KM,
            distance_power=BIAS_DISTANCE_POWER,
            zero_outside_distance_range=(
                BIAS_ZERO_OUTSIDE_DISTANCE_RANGE
            ),
            zero_outside_time_range=(
                BIAS_ZERO_OUTSIDE_TIME_RANGE
            ),
            clip_negative_time_correction=(
                CLIP_NEGATIVE_TIME_CORRECTION
            ),
        )

        # ==================================================
        # Apply bias correction
        # ==================================================
        if APPLY_BIAS_CORRECTION:
            keogram_corrected = (
                keogram_uncorrected
                + bias_correction_keogram
            )

            corrected_dataset_name = (
                "Bias corrected: PCHIP time dependence, "
                f"distance power={BIAS_DISTANCE_POWER:g}"
            )
        else:
            keogram_corrected = (
                keogram_uncorrected.copy()
            )

            corrected_dataset_name = (
                "Bias correction disabled"
            )

        keogram_corrected[
            ~np.isfinite(keogram_uncorrected)
        ] = np.nan

        # ==================================================
        # Select distance rows
        # ==================================================
        selected_distance_mask = (
            (
                distance_km
                >= CORRELATION_DISTANCE_MIN_KM
            )
            & (
                distance_km
                <= CORRELATION_DISTANCE_MAX_KM
            )
        )

        selected_distances = distance_km[
            selected_distance_mask
        ]

        if len(selected_distances) == 0:
            raise ValueError(
                "No rows lie inside the selected distance range."
            )

        # ==================================================
        # Per-angle folders
        # ==================================================
        angle_uncorrected_dir = (
            UNCORRECTED_OUT_DIR
            / f"angle_{angle_deg:03d}"
        )

        angle_corrected_dir = (
            CORRECTED_OUT_DIR
            / f"angle_{angle_deg:03d}"
        )

        angle_uncorrected_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        angle_corrected_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        # ==================================================
        # Uncorrected map
        # ==================================================
        (
            correlation_map_uncorrected,
            selected_distances_uncorrected,
            reference_index_uncorrected,
        ) = calculate_correlation_map(
            keogram=keogram_uncorrected,
            distance_km=distance_km,
            reference_distance_km=0.0,
            selected_distance_mask=selected_distance_mask,
            shifts_pixels=shifts_pixels,
            minimum_points=MIN_VALID_CORRELATION_POINTS,
        )

        (
            best_shift_uncorrected_minutes,
            maximum_correlation_uncorrected,
        ) = find_best_shifts(
            correlation_map=correlation_map_uncorrected,
            shifts_minutes=shifts_minutes,
        )

        uncorrected_slope_results = (
            fit_maximum_correlation_slope(
                distances_km=(
                    selected_distances_uncorrected
                ),
                best_shift_minutes=(
                    best_shift_uncorrected_minutes
                ),
                maximum_correlation=(
                    maximum_correlation_uncorrected
                ),
                minimum_correlation=(
                    MIN_CORRELATION_FOR_SLOPE
                ),
                distance_min_km=(
                    SLOPE_FIT_DISTANCE_MIN_KM
                ),
                distance_max_km=(
                    SLOPE_FIT_DISTANCE_MAX_KM
                ),
            )
        )

        # ==================================================
        # Corrected map
        # ==================================================
        (
            correlation_map_corrected,
            selected_distances_corrected,
            reference_index_corrected,
        ) = calculate_correlation_map(
            keogram=keogram_corrected,
            distance_km=distance_km,
            reference_distance_km=0.0,
            selected_distance_mask=selected_distance_mask,
            shifts_pixels=shifts_pixels,
            minimum_points=MIN_VALID_CORRELATION_POINTS,
        )

        (
            best_shift_corrected_minutes,
            maximum_correlation_corrected,
        ) = find_best_shifts(
            correlation_map=correlation_map_corrected,
            shifts_minutes=shifts_minutes,
        )

        corrected_slope_results = (
            fit_maximum_correlation_slope(
                distances_km=(
                    selected_distances_corrected
                ),
                best_shift_minutes=(
                    best_shift_corrected_minutes
                ),
                maximum_correlation=(
                    maximum_correlation_corrected
                ),
                minimum_correlation=(
                    MIN_CORRELATION_FOR_SLOPE
                ),
                distance_min_km=(
                    SLOPE_FIT_DISTANCE_MIN_KM
                ),
                distance_max_km=(
                    SLOPE_FIT_DISTANCE_MAX_KM
                ),
            )
        )

        # ==================================================
        # Extract 0 km correlation-shift profiles
        # ==================================================
        (
            zero_km_profile_uncorrected,
            zero_km_row_uncorrected,
            zero_km_distance_uncorrected,
        ) = extract_zero_km_correlation_profile(
            correlation_map=correlation_map_uncorrected,
            selected_distances=selected_distances_uncorrected,
        )

        (
            zero_km_profile_corrected,
            zero_km_row_corrected,
            zero_km_distance_corrected,
        ) = extract_zero_km_correlation_profile(
            correlation_map=correlation_map_corrected,
            selected_distances=selected_distances_corrected,
        )

        zero_km_peaks_uncorrected = (
            find_left_and_right_local_maxima(
                correlation_profile=zero_km_profile_uncorrected,
                shifts_minutes=shifts_minutes,
                minimum_prominence=(
                    LOCAL_MAXIMUM_MIN_PROMINENCE
                ),
                minimum_distance_pixels=(
                    LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS
                ),
            )
        )

        zero_km_peaks_corrected = (
            find_left_and_right_local_maxima(
                correlation_profile=zero_km_profile_corrected,
                shifts_minutes=shifts_minutes,
                minimum_prominence=(
                    LOCAL_MAXIMUM_MIN_PROMINENCE
                ),
                minimum_distance_pixels=(
                    LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS
                ),
            )
        )

        # ==================================================
        # Plot maps
        # ==================================================
        uncorrected_plot_file = (
            angle_uncorrected_dir
            / (
                f"correlation_minutes_"
                f"{angle_deg:03d}_uncorrected.png"
            )
        )

        corrected_plot_file = (
            angle_corrected_dir
            / (
                f"correlation_minutes_"
                f"{angle_deg:03d}_corrected.png"
            )
        )

        plot_correlation_map(
            angle_deg=angle_deg,
            full_keogram=keogram_original,
            full_times_min=times_min,
            full_distance_km=distance_km,
            limited_keogram=keogram_uncorrected[
                :,
                selected_distance_mask,
            ],
            limited_times_min=times_correlation_min,
            limited_distance_km=selected_distances_uncorrected,
            correlation_map=(
                correlation_map_uncorrected
            ),
            shifts_minutes=shifts_minutes,
            selected_distances=(
                selected_distances_uncorrected
            ),
            best_shift_minutes=(
                best_shift_uncorrected_minutes
            ),
            maximum_correlation=(
                maximum_correlation_uncorrected
            ),
            slope_results=(
                uncorrected_slope_results
            ),
            zero_km_correlation_profile=(
                zero_km_profile_uncorrected
            ),
            zero_km_distance=(
                zero_km_distance_uncorrected
            ),
            zero_km_local_maxima=(
                zero_km_peaks_uncorrected
            ),
            dataset_name=uncorrected_dataset_name,
            output_file=uncorrected_plot_file,
        )

        plot_correlation_map(
            angle_deg=angle_deg,
            full_keogram=keogram_original,
            full_times_min=times_min,
            full_distance_km=distance_km,
            limited_keogram=keogram_corrected[
                :,
                selected_distance_mask,
            ],
            limited_times_min=times_correlation_min,
            limited_distance_km=selected_distances_corrected,
            correlation_map=(
                correlation_map_corrected
            ),
            shifts_minutes=shifts_minutes,
            selected_distances=(
                selected_distances_corrected
            ),
            best_shift_minutes=(
                best_shift_corrected_minutes
            ),
            maximum_correlation=(
                maximum_correlation_corrected
            ),
            slope_results=(
                corrected_slope_results
            ),
            zero_km_correlation_profile=(
                zero_km_profile_corrected
            ),
            zero_km_distance=(
                zero_km_distance_corrected
            ),
            zero_km_local_maxima=(
                zero_km_peaks_corrected
            ),
            dataset_name=corrected_dataset_name,
            output_file=corrected_plot_file,
        )

        # ==================================================
        # Save arrays
        # ==================================================
        np.save(
            angle_uncorrected_dir
            / (
                f"correlation_"
                f"{angle_deg:03d}_uncorrected.npy"
            ),
            correlation_map_uncorrected,
        )

        np.save(
            angle_uncorrected_dir
            / (
                f"best_shift_minutes_"
                f"{angle_deg:03d}_uncorrected.npy"
            ),
            best_shift_uncorrected_minutes,
        )

        np.save(
            angle_uncorrected_dir
            / (
                f"maximum_correlation_"
                f"{angle_deg:03d}_uncorrected.npy"
            ),
            maximum_correlation_uncorrected,
        )

        np.save(
            angle_corrected_dir
            / (
                f"correlation_"
                f"{angle_deg:03d}_corrected.npy"
            ),
            correlation_map_corrected,
        )

        np.save(
            angle_corrected_dir
            / (
                f"best_shift_minutes_"
                f"{angle_deg:03d}_corrected.npy"
            ),
            best_shift_corrected_minutes,
        )

        np.save(
            angle_corrected_dir
            / (
                f"maximum_correlation_"
                f"{angle_deg:03d}_corrected.npy"
            ),
            maximum_correlation_corrected,
        )

        np.save(
            angle_uncorrected_dir
            / "distances_km.npy",
            selected_distances_uncorrected,
        )

        np.save(
            angle_corrected_dir
            / "distances_km.npy",
            selected_distances_corrected,
        )

        np.save(
            angle_uncorrected_dir
            / "zero_km_correlation_profile.npy",
            zero_km_profile_uncorrected,
        )

        np.save(
            angle_corrected_dir
            / "zero_km_correlation_profile.npy",
            zero_km_profile_corrected,
        )

        zero_km_profile_uncorrected_csv = np.column_stack(
            [
                shifts_minutes,
                zero_km_profile_uncorrected,
            ]
        )

        zero_km_profile_corrected_csv = np.column_stack(
            [
                shifts_minutes,
                zero_km_profile_corrected,
            ]
        )

        np.savetxt(
            angle_uncorrected_dir
            / "zero_km_correlation_profile.csv",
            zero_km_profile_uncorrected_csv,
            delimiter=",",
            header="shift_minutes,correlation",
            comments="",
        )

        np.savetxt(
            angle_corrected_dir
            / "zero_km_correlation_profile.csv",
            zero_km_profile_corrected_csv,
            delimiter=",",
            header="shift_minutes,correlation",
            comments="",
        )

        # ==================================================
        # Save ridge CSV files
        # ==================================================
        uncorrected_summary = np.column_stack(
            [
                selected_distances_uncorrected,
                best_shift_uncorrected_minutes,
                maximum_correlation_uncorrected,
            ]
        )

        corrected_summary = np.column_stack(
            [
                selected_distances_corrected,
                best_shift_corrected_minutes,
                maximum_correlation_corrected,
            ]
        )

        uncorrected_csv = (
            angle_uncorrected_dir
            / (
                f"ridge_"
                f"{angle_deg:03d}_uncorrected.csv"
            )
        )

        corrected_csv = (
            angle_corrected_dir
            / (
                f"ridge_"
                f"{angle_deg:03d}_corrected.csv"
            )
        )

        np.savetxt(
            uncorrected_csv,
            uncorrected_summary,
            delimiter=",",
            header=(
                "distance_km,"
                "best_shift_minutes,"
                "maximum_correlation"
            ),
            comments="",
        )

        np.savetxt(
            corrected_csv,
            corrected_summary,
            delimiter=",",
            header=(
                "distance_km,"
                "best_shift_minutes,"
                "maximum_correlation"
            ),
            comments="",
        )

        # ==================================================
        # Save individual slope report
        # ==================================================
        slope_output_file = (
            SLOPE_REPORT_OUT_DIR
            / (
                f"correlation_slope_"
                f"{angle_deg:03d}.txt"
            )
        )

        with open(
            slope_output_file,
            "w",
            encoding="utf-8",
        ) as file:
            file.write(
                f"Angle: {angle_deg} degrees\n"
            )

            file.write(
                f"Minutes per pixel: "
                f"{MINUTES_PER_PIXEL:.10f}\n\n"
            )

            file.write(
                "UNCORRECTED\n"
            )

            for key, value in (
                uncorrected_slope_results.items()
            ):
                if key != "fit_mask":
                    file.write(
                        f"{key}: {value}\n"
                    )

            file.write(
                "\nCORRECTED\n"
            )

            for key, value in (
                corrected_slope_results.items()
            ):
                if key != "fit_mask":
                    file.write(
                        f"{key}: {value}\n"
                    )

            file.write(
                "\n0 KM CORRELATION-PROFILE LOCAL MAXIMA\n"
            )

            file.write(
                "Uncorrected left shift (min): "
                f"{zero_km_peaks_uncorrected['left']['shift_minutes']}\n"
            )
            file.write(
                "Uncorrected left correlation: "
                f"{zero_km_peaks_uncorrected['left']['correlation']}\n"
            )
            file.write(
                "Uncorrected right shift (min): "
                f"{zero_km_peaks_uncorrected['right']['shift_minutes']}\n"
            )
            file.write(
                "Uncorrected right correlation: "
                f"{zero_km_peaks_uncorrected['right']['correlation']}\n"
            )
            file.write(
                "Corrected left shift (min): "
                f"{zero_km_peaks_corrected['left']['shift_minutes']}\n"
            )
            file.write(
                "Corrected left correlation: "
                f"{zero_km_peaks_corrected['left']['correlation']}\n"
            )
            file.write(
                "Corrected right shift (min): "
                f"{zero_km_peaks_corrected['right']['shift_minutes']}\n"
            )
            file.write(
                "Corrected right correlation: "
                f"{zero_km_peaks_corrected['right']['correlation']}\n"
            )

        # ==================================================
        # Add to combined summary
        # ==================================================
        combined_results.append(
            [
                float(angle_deg),

                uncorrected_slope_results[
                    "slope_min_per_km"
                ],
                uncorrected_slope_results[
                    "intercept_minutes"
                ],
                uncorrected_slope_results[
                    "speed_km_per_min"
                ],
                uncorrected_slope_results[
                    "speed_m_per_s"
                ],
                uncorrected_slope_results[
                    "r_squared"
                ],
                uncorrected_slope_results[
                    "number_of_fit_points"
                ],

                corrected_slope_results[
                    "slope_min_per_km"
                ],
                corrected_slope_results[
                    "intercept_minutes"
                ],
                corrected_slope_results[
                    "speed_km_per_min"
                ],
                corrected_slope_results[
                    "speed_m_per_s"
                ],
                corrected_slope_results[
                    "r_squared"
                ],
                corrected_slope_results[
                    "number_of_fit_points"
                ],
            ]
        )

        combined_zero_km_peak_results.append(
            [
                float(angle_deg),
                zero_km_peaks_uncorrected["left"][
                    "shift_minutes"
                ],
                zero_km_peaks_uncorrected["left"][
                    "correlation"
                ],
                zero_km_peaks_uncorrected["right"][
                    "shift_minutes"
                ],
                zero_km_peaks_uncorrected["right"][
                    "correlation"
                ],
                zero_km_peaks_corrected["left"][
                    "shift_minutes"
                ],
                zero_km_peaks_corrected["left"][
                    "correlation"
                ],
                zero_km_peaks_corrected["right"][
                    "shift_minutes"
                ],
                zero_km_peaks_corrected["right"][
                    "correlation"
                ],
            ]
        )

        successful_angles.append(
            int(angle_deg)
        )

        print_slope_results(
            angle_deg=angle_deg,
            label=(
                "Uncorrected maximum-correlation ridge"
            ),
            results=uncorrected_slope_results,
        )

        print_slope_results(
            angle_deg=angle_deg,
            label=(
                "Corrected maximum-correlation ridge"
            ),
            results=corrected_slope_results,
        )

        print(
            "\n0 km correlation-profile local maxima:"
        )
        print(
            "  Uncorrected left: "
            f"{zero_km_peaks_uncorrected['left']['shift_minutes']:.3f} min, "
            f"r={zero_km_peaks_uncorrected['left']['correlation']:.6f}"
        )
        print(
            "  Uncorrected right: "
            f"{zero_km_peaks_uncorrected['right']['shift_minutes']:.3f} min, "
            f"r={zero_km_peaks_uncorrected['right']['correlation']:.6f}"
        )
        print(
            "  Corrected left: "
            f"{zero_km_peaks_corrected['left']['shift_minutes']:.3f} min, "
            f"r={zero_km_peaks_corrected['left']['correlation']:.6f}"
        )
        print(
            "  Corrected right: "
            f"{zero_km_peaks_corrected['right']['shift_minutes']:.3f} min, "
            f"r={zero_km_peaks_corrected['right']['correlation']:.6f}"
        )

    except Exception as error:
        print(
            f"ERROR processing {angle_deg}°:"
        )

        print(
            error
        )

        failed_angles.append(
            (
                int(angle_deg),
                str(error),
            )
        )


# ==================================================
# Save combined 0 km local-maximum summary
# ==================================================
combined_zero_km_peak_results = np.asarray(
    combined_zero_km_peak_results,
    dtype=float,
)

combined_zero_km_peak_csv = (
    OUT_DIR
    / "all_angle_zero_km_local_maxima.csv"
)

combined_zero_km_peak_npy = (
    OUT_DIR
    / "all_angle_zero_km_local_maxima.npy"
)

combined_zero_km_peak_header = (
    "angle_deg,"
    "uncorrected_left_shift_minutes,"
    "uncorrected_left_correlation,"
    "uncorrected_right_shift_minutes,"
    "uncorrected_right_correlation,"
    "corrected_left_shift_minutes,"
    "corrected_left_correlation,"
    "corrected_right_shift_minutes,"
    "corrected_right_correlation"
)

if combined_zero_km_peak_results.size > 0:
    np.savetxt(
        combined_zero_km_peak_csv,
        combined_zero_km_peak_results,
        delimiter=",",
        header=combined_zero_km_peak_header,
        comments="",
    )

    np.save(
        combined_zero_km_peak_npy,
        combined_zero_km_peak_results,
    )


# ==================================================
# Save combined angle summary
# ==================================================
combined_results = np.asarray(
    combined_results,
    dtype=float,
)

combined_csv = (
    OUT_DIR
    / "all_angle_slope_summary.csv"
)

combined_npy = (
    OUT_DIR
    / "all_angle_slope_summary.npy"
)

combined_header = (
    "angle_deg,"
    "uncorrected_slope_min_per_km,"
    "uncorrected_intercept_minutes,"
    "uncorrected_speed_km_per_min,"
    "uncorrected_speed_m_per_s,"
    "uncorrected_r_squared,"
    "uncorrected_number_of_fit_points,"
    "corrected_slope_min_per_km,"
    "corrected_intercept_minutes,"
    "corrected_speed_km_per_min,"
    "corrected_speed_m_per_s,"
    "corrected_r_squared,"
    "corrected_number_of_fit_points"
)

if combined_results.size > 0:
    np.savetxt(
        combined_csv,
        combined_results,
        delimiter=",",
        header=combined_header,
        comments="",
    )

    np.save(
        combined_npy,
        combined_results,
    )


# ==================================================
# Plot speed versus angle
# ==================================================
if combined_results.size > 0:
    angles_processed = combined_results[
        :,
        0,
    ]

    uncorrected_speed_m_per_s = combined_results[
        :,
        4,
    ]

    corrected_speed_m_per_s = combined_results[
        :,
        10,
    ]

    fig_speed, ax_speed = plt.subplots(
        figsize=(10, 5)
    )

    ax_speed.plot(
        angles_processed,
        uncorrected_speed_m_per_s,
        marker="o",
        linewidth=1.5,
        label="Uncorrected",
    )

    ax_speed.plot(
        angles_processed,
        corrected_speed_m_per_s,
        marker="o",
        linewidth=1.5,
        label="Corrected",
    )

    ax_speed.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
    )

    ax_speed.set_xlim(
        KEOGRAM_ANGLES_DEG.min(),
        KEOGRAM_ANGLES_DEG.max(),
    )

    ax_speed.set_xlabel(
        "Keogram angle (degrees)"
    )

    ax_speed.set_ylabel(
        "Equivalent signed speed (m/s)"
    )

    ax_speed.set_title(
        "Maximum-Correlation Ridge Speed versus Angle"
    )

    ax_speed.grid(
        True,
        alpha=0.3,
    )

    ax_speed.legend(
        loc="best",
    )

    plt.tight_layout()

    speed_angle_plot = (
        OUT_DIR
        / "speed_vs_angle.png"
    )

    fig_speed.savefig(
        speed_angle_plot,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig_speed
    )


# ==================================================
# Plot R-squared versus angle
# ==================================================
if combined_results.size > 0:
    uncorrected_r_squared = combined_results[
        :,
        5,
    ]

    corrected_r_squared = combined_results[
        :,
        11,
    ]

    fig_r2, ax_r2 = plt.subplots(
        figsize=(10, 5)
    )

    ax_r2.plot(
        angles_processed,
        uncorrected_r_squared,
        marker="o",
        linewidth=1.5,
        label="Uncorrected",
    )

    ax_r2.plot(
        angles_processed,
        corrected_r_squared,
        marker="o",
        linewidth=1.5,
        label="Corrected",
    )

    ax_r2.set_xlim(
        KEOGRAM_ANGLES_DEG.min(),
        KEOGRAM_ANGLES_DEG.max(),
    )

    ax_r2.set_ylim(
        0.0,
        1.0,
    )

    ax_r2.set_xlabel(
        "Keogram angle (degrees)"
    )

    ax_r2.set_ylabel(
        "R²"
    )

    ax_r2.set_title(
        "Maximum-Correlation Ridge Fit R² versus Angle"
    )

    ax_r2.grid(
        True,
        alpha=0.3,
    )

    ax_r2.legend(
        loc="best",
    )

    plt.tight_layout()

    r2_angle_plot = (
        OUT_DIR
        / "r_squared_vs_angle.png"
    )

    fig_r2.savefig(
        r2_angle_plot,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig_r2
    )


# ==================================================
# Save processing log
# ==================================================
processing_log_file = (
    OUT_DIR
    / "processing_log.txt"
)

with open(
    processing_log_file,
    "w",
    encoding="utf-8",
) as file:
    file.write(
        "Requested angles:\n"
    )

    file.write(
        ", ".join(
            str(angle)
            for angle in KEOGRAM_ANGLES_DEG
        )
        + "\n\n"
    )

    file.write(
        f"Number requested: "
        f"{len(KEOGRAM_ANGLES_DEG)}\n"
    )

    file.write(
        f"Number successful: "
        f"{len(successful_angles)}\n"
    )

    file.write(
        f"Number failed: "
        f"{len(failed_angles)}\n\n"
    )

    file.write(
        "Successful angles:\n"
    )

    file.write(
        ", ".join(
            str(angle)
            for angle in successful_angles
        )
        + "\n\n"
    )

    file.write(
        "Failed angles:\n"
    )

    if failed_angles:
        for angle_deg, reason in failed_angles:
            file.write(
                f"{angle_deg}: {reason}\n"
            )
    else:
        file.write(
            "None\n"
        )


# ==================================================
# Print final summary
# ==================================================
print(
    "\n"
    + "=" * 70
)

print(
    "ALL-ANGLE PROCESSING COMPLETE"
)

print(
    "=" * 70
)

print(
    "Angles requested:"
)

print(
    KEOGRAM_ANGLES_DEG
)

print(
    f"\nNumber requested: "
    f"{len(KEOGRAM_ANGLES_DEG)}"
)

print(
    f"Number successful: "
    f"{len(successful_angles)}"
)

print(
    f"Number failed: "
    f"{len(failed_angles)}"
)

print(
    "\nSuccessful angles:"
)

print(
    successful_angles
)

if failed_angles:
    print(
        "\nFailed angles:"
    )

    for angle_deg, reason in failed_angles:
        print(
            f"  {angle_deg}°: {reason}"
        )

print(
    "\nCombined CSV:"
)

print(
    combined_csv
)

print(
    "\nOutput folder:"
)

print(
    OUT_DIR.resolve()
)

Angles to process:
[  0   5  10  15  20  25  30  35  40  45  50  55  60  65  70  75  80  85
  90  95 100 105 110 115 120 125 130 135 140 145 150 155 160 165 170 175]
Number of angles: 36

Time sampling:
One time pixel = 2.000000 minutes
One time pixel = 120.000 seconds
Maximum spacing deviation = 0.000000e+00 minutes

Shift range:
Requested: -200.00 to 200.00 min
Actual: -200.00 to 200.00 min
Pixel shifts: -100 to 100

Processing angle 0° (1/36)

0° — Uncorrected maximum-correlation ridge:
  Shift slope: 0.02600038 min/km
  Intercept: -0.054828 min
  Equivalent speed: 641.0163 m/s
  R²: 0.862005
  Points used: 251

0° — Corrected maximum-correlation ridge:
  Shift slope: 0.02600038 min/km
  Intercept: -0.054828 min
  Equivalent speed: 641.0163 m/s
  R²: 0.862005
  Points used: 251

0 km correlation-profile local maxima:
  Uncorrected left: -164.000 min, r=0.494851
  Uncorrected right: 164.000 min, r=0.494851
  Corrected left: -164.000 min, r=0.494851
  Corrected right: 164.000 min, r=0

## Smaller keogram range

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.ndimage import gaussian_filter
from scipy.interpolate import PchipInterpolator
from scipy.signal import find_peaks


# ==================================================
# Settings
# ==================================================
BASE_OUT_DIR = Path(
    "BLO_perturbation_keogram_outputs_new"
)

# Process every directional keogram:
# 0, 5, 10, ..., 175 degrees
KEOGRAM_ANGLES_DEG = np.arange(
    0,
    176,
    5,
    dtype=int,
)

# Time range used for correlation
CORRELATION_TIME_MIN = 0.0
CORRELATION_TIME_MAX = 70.0

# Distance range included in the correlation map
CORRELATION_DISTANCE_MIN_KM = 0.0
CORRELATION_DISTANCE_MAX_KM = 250.0

# Shift range in minutes
SHIFT_MIN_MINUTES = -60.0
SHIFT_MAX_MINUTES = 60.0

# Minimum number of overlapping time samples
MIN_VALID_CORRELATION_POINTS = 10

# Minimum maximum-correlation value used in the slope fit
MIN_CORRELATION_FOR_SLOPE = 0.50

# Distance range used in the ridge fit
SLOPE_FIT_DISTANCE_MIN_KM = 0.0
SLOPE_FIT_DISTANCE_MAX_KM = 250.0

DPI = 200

CORRELATION_CMAP = "RdBu_r"
KEOGRAM_CMAP = "RdBu_r"

# Use symmetric automatic keogram limits based on this percentile.
# Set KEOGRAM_ABS_LIMIT to a number to force fixed limits.
KEOGRAM_ABS_LIMIT = None
KEOGRAM_AUTO_PERCENTILE = 99.0

# Correlation color scale
CORRELATION_VMIN = 0.00
CORRELATION_VMAX = 1.00

# Local maxima search on the 0 km correlation-shift profile.
# The left maximum is searched at negative shifts and the right
# maximum is searched at positive shifts. The zero-shift point is
# therefore excluded automatically.
LOCAL_MAXIMUM_MIN_PROMINENCE = 0.0
LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS = 1


# ==================================================
# Optional 2D smoothing
# ==================================================
APPLY_2D_SMOOTHING = True

# Keogram dimensions:
# axis 0 = time
# axis 1 = distance
SMOOTH_SIGMA_TIME_PIXELS = 1.5
SMOOTH_SIGMA_DISTANCE_PIXELS = 2.0

SMOOTH_MODE = "nearest"
SMOOTH_WEIGHT_TOLERANCE = 1e-12


# ==================================================
# Bias-correction settings
# ==================================================
APPLY_BIAS_CORRECTION = False

BIAS_TIME_POINTS_MIN = np.array(
    [
        0.0,
        150.0,
        300.0,
    ],
    dtype=float,
)

BIAS_AT_MAX_DISTANCE = np.array(
    [
        0.20,
        0.04,
        -0.01,
    ],
    dtype=float,
)

BIAS_DISTANCE_MIN_KM = 0.0
BIAS_DISTANCE_MAX_KM = 250.0

# 1.0 = linear distance dependence
# 2.0 = quadratic distance dependence
BIAS_DISTANCE_POWER = 1.0

BIAS_ZERO_OUTSIDE_DISTANCE_RANGE = True
BIAS_ZERO_OUTSIDE_TIME_RANGE = True
CLIP_NEGATIVE_TIME_CORRECTION = False


# ==================================================
# Filename-safe numeric label
# ==================================================
def safe_number_label(value):
    value_float = float(value)

    if value_float.is_integer():
        return str(
            int(value_float)
        )

    return (
        f"{value_float:g}"
        .replace("-", "m")
        .replace(".", "p")
    )


# ==================================================
# NaN-aware 2D Gaussian smoothing
# ==================================================
def smooth_keogram_2d(
    data,
    sigma_time,
    sigma_distance,
    mode="nearest",
    weight_tolerance=1e-12,
):
    """
    Apply NaN-aware Gaussian smoothing to a 2D keogram.
    """
    data = np.asarray(
        data,
        dtype=float,
    )

    if data.ndim != 2:
        raise ValueError(
            "The keogram must be two-dimensional."
        )

    if sigma_time < 0 or sigma_distance < 0:
        raise ValueError(
            "Smoothing sigma values must be nonnegative."
        )

    if sigma_time == 0 and sigma_distance == 0:
        return data.copy()

    valid_mask = np.isfinite(
        data
    )

    data_filled = np.where(
        valid_mask,
        data,
        0.0,
    )

    weights = valid_mask.astype(
        float
    )

    sigma = (
        sigma_time,
        sigma_distance,
    )

    smoothed_values = gaussian_filter(
        data_filled,
        sigma=sigma,
        mode=mode,
    )

    smoothed_weights = gaussian_filter(
        weights,
        sigma=sigma,
        mode=mode,
    )

    smoothed = np.full(
        data.shape,
        np.nan,
        dtype=float,
    )

    valid_output = (
        smoothed_weights
        > weight_tolerance
    )

    smoothed[valid_output] = (
        smoothed_values[valid_output]
        / smoothed_weights[valid_output]
    )

    return smoothed


# ==================================================
# Create 2D bias-correction keogram
# ==================================================
def create_bias_correction_keogram(
    times_min,
    distance_km,
    bias_time_points_min,
    bias_at_max_distance,
    bias_distance_min_km,
    bias_distance_max_km,
    distance_power=1.0,
    zero_outside_distance_range=True,
    zero_outside_time_range=True,
    clip_negative_time_correction=False,
):
    """
    Create the time x distance additive bias field.
    """
    times_min = np.asarray(
        times_min,
        dtype=float,
    )

    distance_km = np.asarray(
        distance_km,
        dtype=float,
    )

    bias_time_points_min = np.asarray(
        bias_time_points_min,
        dtype=float,
    )

    bias_at_max_distance = np.asarray(
        bias_at_max_distance,
        dtype=float,
    )

    if (
        len(bias_time_points_min)
        != len(bias_at_max_distance)
    ):
        raise ValueError(
            "Bias time points and bias values must have "
            "the same length."
        )

    if not np.all(
        np.diff(bias_time_points_min) > 0
    ):
        raise ValueError(
            "BIAS_TIME_POINTS_MIN must be strictly increasing."
        )

    if bias_distance_max_km <= bias_distance_min_km:
        raise ValueError(
            "The maximum bias distance must exceed "
            "the minimum bias distance."
        )

    if distance_power <= 0:
        raise ValueError(
            "BIAS_DISTANCE_POWER must be positive."
        )

    # ==================================================
    # PCHIP time dependence
    # ==================================================
    interpolator = PchipInterpolator(
        bias_time_points_min,
        bias_at_max_distance,
        extrapolate=False,
    )

    bias_vs_time = interpolator(
        times_min
    )

    if zero_outside_time_range:
        inside_time = (
            (times_min >= bias_time_points_min[0])
            & (times_min <= bias_time_points_min[-1])
        )

        bias_vs_time = np.where(
            inside_time,
            bias_vs_time,
            0.0,
        )
    else:
        bias_vs_time = np.where(
            times_min < bias_time_points_min[0],
            bias_at_max_distance[0],
            bias_vs_time,
        )

        bias_vs_time = np.where(
            times_min > bias_time_points_min[-1],
            bias_at_max_distance[-1],
            bias_vs_time,
        )

    if clip_negative_time_correction:
        bias_vs_time = np.maximum(
            bias_vs_time,
            0.0,
        )

    # ==================================================
    # Distance dependence
    # ==================================================
    normalized_distance = (
        distance_km
        - bias_distance_min_km
    ) / (
        bias_distance_max_km
        - bias_distance_min_km
    )

    if zero_outside_distance_range:
        inside_distance = (
            (distance_km >= bias_distance_min_km)
            & (distance_km <= bias_distance_max_km)
        )

        normalized_distance = np.where(
            inside_distance,
            normalized_distance,
            0.0,
        )
    else:
        normalized_distance = np.clip(
            normalized_distance,
            0.0,
            1.0,
        )

    distance_factor = (
        normalized_distance ** distance_power
    )

    correction_keogram = (
        bias_vs_time[:, np.newaxis]
        * distance_factor[np.newaxis, :]
    )

    return (
        correction_keogram,
        bias_vs_time,
        distance_factor,
    )


# ==================================================
# NaN-aware Pearson correlation
# ==================================================
def nan_pearson_correlation(
    reference,
    comparison,
    minimum_points=20,
):
    """
    Calculate Pearson correlation using finite overlapping values.
    """
    reference = np.asarray(
        reference,
        dtype=float,
    )

    comparison = np.asarray(
        comparison,
        dtype=float,
    )

    if reference.shape != comparison.shape:
        raise ValueError(
            "The two arrays must have the same shape."
        )

    valid = (
        np.isfinite(reference)
        & np.isfinite(comparison)
    )

    if np.count_nonzero(valid) < minimum_points:
        return np.nan

    reference_valid = reference[
        valid
    ]

    comparison_valid = comparison[
        valid
    ]

    reference_centered = (
        reference_valid
        - np.mean(reference_valid)
    )

    comparison_centered = (
        comparison_valid
        - np.mean(comparison_valid)
    )

    denominator = np.sqrt(
        np.sum(reference_centered ** 2)
        * np.sum(comparison_centered ** 2)
    )

    if (
        not np.isfinite(denominator)
        or denominator <= 0
    ):
        return np.nan

    correlation = (
        np.sum(
            reference_centered
            * comparison_centered
        )
        / denominator
    )

    return np.clip(
        correlation,
        -1.0,
        1.0,
    )


# ==================================================
# Correlation for one integer-pixel shift
# ==================================================
def calculate_shifted_correlation(
    reference_row,
    comparison_row,
    shift_pixels,
    minimum_points=20,
):
    """
    Shift comparison_row and correlate it with reference_row.

    Positive shift:
        comparison row moves toward later times.

    Negative shift:
        comparison row moves toward earlier times.

    No circular wrapping is used.
    """
    reference_row = np.asarray(
        reference_row,
        dtype=float,
    )

    comparison_row = np.asarray(
        comparison_row,
        dtype=float,
    )

    if reference_row.shape != comparison_row.shape:
        raise ValueError(
            "Reference and comparison rows must have "
            "the same shape."
        )

    ntime = len(
        reference_row
    )

    if abs(shift_pixels) >= ntime:
        return np.nan

    if shift_pixels > 0:
        reference_overlap = reference_row[
            shift_pixels:
        ]

        comparison_overlap = comparison_row[
            :-shift_pixels
        ]

    elif shift_pixels < 0:
        shift_abs = abs(
            shift_pixels
        )

        reference_overlap = reference_row[
            :-shift_abs
        ]

        comparison_overlap = comparison_row[
            shift_abs:
        ]

    else:
        reference_overlap = reference_row
        comparison_overlap = comparison_row

    return nan_pearson_correlation(
        reference=reference_overlap,
        comparison=comparison_overlap,
        minimum_points=minimum_points,
    )


# ==================================================
# Calculate complete correlation map
# ==================================================
def calculate_correlation_map(
    keogram,
    distance_km,
    reference_distance_km,
    selected_distance_mask,
    shifts_pixels,
    minimum_points=20,
):
    """
    Correlate every selected distance row with the reference row
    for every requested shift.
    """
    keogram = np.asarray(
        keogram,
        dtype=float,
    )

    distance_km = np.asarray(
        distance_km,
        dtype=float,
    )

    shifts_pixels = np.asarray(
        shifts_pixels,
        dtype=int,
    )

    reference_index = np.argmin(
        np.abs(
            distance_km
            - reference_distance_km
        )
    )

    selected_indices = np.where(
        selected_distance_mask
    )[0]

    selected_distances = distance_km[
        selected_indices
    ]

    reference_row = keogram[
        :,
        reference_index,
    ]

    correlation_map = np.full(
        (
            len(selected_indices),
            len(shifts_pixels),
        ),
        np.nan,
        dtype=float,
    )

    for output_row_index, keogram_row_index in enumerate(
        selected_indices
    ):
        comparison_row = keogram[
            :,
            keogram_row_index,
        ]

        for shift_index, shift_pixels in enumerate(
            shifts_pixels
        ):
            correlation_map[
                output_row_index,
                shift_index,
            ] = calculate_shifted_correlation(
                reference_row=reference_row,
                comparison_row=comparison_row,
                shift_pixels=int(shift_pixels),
                minimum_points=minimum_points,
            )

    return (
        correlation_map,
        selected_distances,
        reference_index,
    )


# ==================================================
# Maximum-correlation shift for each row
# ==================================================
def find_best_shifts(
    correlation_map,
    shifts_minutes,
):
    """
    Find the shift in minutes having the largest correlation
    for every distance row.
    """
    correlation_map = np.asarray(
        correlation_map,
        dtype=float,
    )

    shifts_minutes = np.asarray(
        shifts_minutes,
        dtype=float,
    )

    best_shift_minutes = np.full(
        correlation_map.shape[0],
        np.nan,
        dtype=float,
    )

    maximum_correlation = np.full(
        correlation_map.shape[0],
        np.nan,
        dtype=float,
    )

    for row_index in range(
        correlation_map.shape[0]
    ):
        correlation_row = correlation_map[
            row_index,
            :
        ]

        finite = np.isfinite(
            correlation_row
        )

        if not np.any(finite):
            continue

        finite_indices = np.where(
            finite
        )[0]

        local_max_index = np.argmax(
            correlation_row[finite]
        )

        full_max_index = finite_indices[
            local_max_index
        ]

        best_shift_minutes[row_index] = (
            shifts_minutes[full_max_index]
        )

        maximum_correlation[row_index] = (
            correlation_row[full_max_index]
        )

    return (
        best_shift_minutes,
        maximum_correlation,
    )


# ==================================================
# Extract the correlation-shift profile at 0 km
# ==================================================
def extract_zero_km_correlation_profile(
    correlation_map,
    selected_distances,
):
    """
    Return the correlation-versus-shift profile for the selected
    distance row nearest 0 km.
    """
    correlation_map = np.asarray(
        correlation_map,
        dtype=float,
    )

    selected_distances = np.asarray(
        selected_distances,
        dtype=float,
    )

    if correlation_map.ndim != 2:
        raise ValueError(
            "correlation_map must be two-dimensional."
        )

    if correlation_map.shape[0] != len(selected_distances):
        raise ValueError(
            "The number of correlation-map rows must equal "
            "the number of selected distances."
        )

    zero_distance_row_index = int(
        np.argmin(
            np.abs(selected_distances)
        )
    )

    zero_distance_km = float(
        selected_distances[zero_distance_row_index]
    )

    zero_km_profile = correlation_map[
        zero_distance_row_index,
        :
    ].copy()

    return (
        zero_km_profile,
        zero_distance_row_index,
        zero_distance_km,
    )


# ==================================================
# Find strongest local maximum on each side of zero
# ==================================================
def find_left_and_right_local_maxima(
    correlation_profile,
    shifts_minutes,
    minimum_prominence=0.0,
    minimum_distance_pixels=1,
):
    """
    Find the strongest finite local maximum at negative shift
    and the strongest finite local maximum at positive shift.

    The point at zero shift is not included in either search.

    If scipy.signal.find_peaks finds no strict local maximum on one
    side, the largest finite value on that side is used as a fallback.
    """
    correlation_profile = np.asarray(
        correlation_profile,
        dtype=float,
    )

    shifts_minutes = np.asarray(
        shifts_minutes,
        dtype=float,
    )

    if correlation_profile.shape != shifts_minutes.shape:
        raise ValueError(
            "correlation_profile and shifts_minutes must "
            "have the same shape."
        )

    def find_one_side(side_mask):
        side_indices = np.where(
            side_mask
            & np.isfinite(correlation_profile)
        )[0]

        if len(side_indices) == 0:
            return {
                "index": -1,
                "shift_minutes": np.nan,
                "correlation": np.nan,
                "used_fallback": False,
            }

        side_values = correlation_profile[
            side_indices
        ]

        local_indices, properties = find_peaks(
            side_values,
            prominence=minimum_prominence,
            distance=minimum_distance_pixels,
        )

        used_fallback = False

        if len(local_indices) > 0:
            local_values = side_values[
                local_indices
            ]

            strongest_local_position = local_indices[
                int(np.nanargmax(local_values))
            ]

            full_index = int(
                side_indices[strongest_local_position]
            )
        else:
            used_fallback = True

            strongest_side_position = int(
                np.nanargmax(side_values)
            )

            full_index = int(
                side_indices[strongest_side_position]
            )

        return {
            "index": full_index,
            "shift_minutes": float(
                shifts_minutes[full_index]
            ),
            "correlation": float(
                correlation_profile[full_index]
            ),
            "used_fallback": used_fallback,
        }

    left_result = find_one_side(
        shifts_minutes < 0.0
    )

    right_result = find_one_side(
        shifts_minutes > 0.0
    )

    return {
        "left": left_result,
        "right": right_result,
    }


# ==================================================
# Fit maximum-correlation ridge
# ==================================================
def fit_maximum_correlation_slope(
    distances_km,
    best_shift_minutes,
    maximum_correlation,
    minimum_correlation=0.5,
    distance_min_km=None,
    distance_max_km=None,
):
    """
    Fit:

        shift_minutes = slope * distance_km + intercept
    """
    distances_km = np.asarray(
        distances_km,
        dtype=float,
    )

    best_shift_minutes = np.asarray(
        best_shift_minutes,
        dtype=float,
    )

    maximum_correlation = np.asarray(
        maximum_correlation,
        dtype=float,
    )

    fit_mask = (
        np.isfinite(distances_km)
        & np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (
            maximum_correlation
            >= minimum_correlation
        )
    )

    if distance_min_km is not None:
        fit_mask &= (
            distances_km
            >= distance_min_km
        )

    if distance_max_km is not None:
        fit_mask &= (
            distances_km
            <= distance_max_km
        )

    number_of_points = np.count_nonzero(
        fit_mask
    )

    if number_of_points < 2:
        return {
            "slope_min_per_km": np.nan,
            "intercept_minutes": np.nan,
            "speed_km_per_min": np.nan,
            "speed_m_per_s": np.nan,
            "r_squared": np.nan,
            "number_of_fit_points": number_of_points,
            "fit_mask": fit_mask,
        }

    fit_distances = distances_km[
        fit_mask
    ]

    fit_shifts = best_shift_minutes[
        fit_mask
    ]

    slope_min_per_km, intercept_minutes = np.polyfit(
        fit_distances,
        fit_shifts,
        deg=1,
    )

    predicted_shifts = (
        slope_min_per_km
        * fit_distances
        + intercept_minutes
    )

    residual_sum_squares = np.sum(
        (
            fit_shifts
            - predicted_shifts
        ) ** 2
    )

    total_sum_squares = np.sum(
        (
            fit_shifts
            - np.mean(fit_shifts)
        ) ** 2
    )

    if total_sum_squares > 0:
        r_squared = (
            1.0
            - residual_sum_squares
            / total_sum_squares
        )
    else:
        r_squared = np.nan

    if np.isclose(
        slope_min_per_km,
        0.0,
    ):
        speed_km_per_min = np.nan
        speed_m_per_s = np.nan
    else:
        speed_km_per_min = (
            1.0
            / slope_min_per_km
        )

        speed_m_per_s = (
            speed_km_per_min
            * 1000.0
            / 60.0
        )

    return {
        "slope_min_per_km": slope_min_per_km,
        "intercept_minutes": intercept_minutes,
        "speed_km_per_min": speed_km_per_min,
        "speed_m_per_s": speed_m_per_s,
        "r_squared": r_squared,
        "number_of_fit_points": number_of_points,
        "fit_mask": fit_mask,
    }


# ==================================================
# Keogram color limit
# ==================================================
def determine_keogram_abs_limit(
    full_keogram,
    limited_keogram,
):
    """
    Return one symmetric absolute color limit shared by both
    keogram panels.
    """
    if KEOGRAM_ABS_LIMIT is not None:
        abs_limit = float(
            KEOGRAM_ABS_LIMIT
        )

        if (
            not np.isfinite(abs_limit)
            or abs_limit <= 0
        ):
            raise ValueError(
                "KEOGRAM_ABS_LIMIT must be positive and finite."
            )

        return abs_limit

    finite_values = []

    for data in [
        full_keogram,
        limited_keogram,
    ]:
        data = np.asarray(
            data,
            dtype=float,
        )

        finite = np.abs(
            data[np.isfinite(data)]
        )

        if finite.size > 0:
            finite_values.append(
                finite
            )

    if len(finite_values) == 0:
        return 1.0

    all_finite = np.concatenate(
        finite_values
    )

    abs_limit = np.nanpercentile(
        all_finite,
        KEOGRAM_AUTO_PERCENTILE,
    )

    if (
        not np.isfinite(abs_limit)
        or abs_limit <= 0
    ):
        abs_limit = np.nanmax(
            all_finite
        )

    if (
        not np.isfinite(abs_limit)
        or abs_limit <= 0
    ):
        abs_limit = 1.0

    return float(
        abs_limit
    )


# ==================================================
# Plot keograms, correlation map, ridge, and profile
# ==================================================
def plot_correlation_map(
    angle_deg,
    full_keogram,
    full_times_min,
    full_distance_km,
    limited_keogram,
    limited_times_min,
    limited_distance_km,
    correlation_map,
    shifts_minutes,
    selected_distances,
    best_shift_minutes,
    maximum_correlation,
    slope_results,
    zero_km_correlation_profile,
    zero_km_distance,
    zero_km_local_maxima,
    dataset_name,
    output_file,
):
    """
    Plot:
      1. full-range original keogram;
      2. limited-range smoothed/corrected keogram;
      3. correlation versus time shift and distance;
      4. correlation versus shift for the 0 km row.
    """
    fig, (
        ax_full,
        ax_limited,
        ax_map,
        ax_profile,
    ) = plt.subplots(
        nrows=4,
        ncols=1,
        figsize=(11, 16),
        gridspec_kw={
            "height_ratios": [
                1.55,
                1.55,
                2.5,
                1.15,
            ],
        },
    )

    keogram_abs_limit = determine_keogram_abs_limit(
        full_keogram=full_keogram,
        limited_keogram=limited_keogram,
    )

    # ==================================================
    # Full-range original keogram
    # ==================================================
    full_mesh = ax_full.pcolormesh(
        full_times_min,
        full_distance_km,
        full_keogram.T,
        shading="auto",
        cmap=KEOGRAM_CMAP,
        vmin=-keogram_abs_limit,
        vmax=keogram_abs_limit,
    )

    full_cbar = fig.colorbar(
        full_mesh,
        ax=ax_full,
        fraction=0.025,
        pad=0.02,
    )

    full_cbar.set_label(
        "Keogram value"
    )

    ax_full.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
    )

    ax_full.set_xlim(
        np.nanmin(full_times_min),
        np.nanmax(full_times_min),
    )

    ax_full.set_ylim(
        np.nanmin(full_distance_km),
        np.nanmax(full_distance_km),
    )

    ax_full.set_title(
        f"{angle_deg}° Full-Range Original Keogram"
    )

    ax_full.set_xlabel(
        "Time (min)"
    )

    ax_full.set_ylabel(
        "Distance along slice (km)"
    )

    # ==================================================
    # Limited-range processed keogram
    # ==================================================
    limited_mesh = ax_limited.pcolormesh(
        limited_times_min,
        limited_distance_km,
        limited_keogram.T,
        shading="auto",
        cmap=KEOGRAM_CMAP,
        #vmin=-keogram_abs_limit,
        #vmax=keogram_abs_limit,
    )

    limited_cbar = fig.colorbar(
        limited_mesh,
        ax=ax_limited,
        fraction=0.025,
        pad=0.02,
    )

    limited_cbar.set_label(
        "Keogram value"
    )

    ax_limited.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
    )

    ax_limited.set_xlim(
        CORRELATION_TIME_MIN,
        CORRELATION_TIME_MAX,
    )

    ax_limited.set_ylim(
        CORRELATION_DISTANCE_MIN_KM,
        CORRELATION_DISTANCE_MAX_KM,
    )

    ax_limited.set_title(
        "Limited-Range Processed Keogram\n"
        f"{dataset_name}"
    )

    ax_limited.set_xlabel(
        "Time (min)"
    )

    ax_limited.set_ylabel(
        "Distance along slice (km)"
    )

    # ==================================================
    # Correlation map
    # ==================================================
    if len(shifts_minutes) > 1:
        shift_spacing = np.median(
            np.diff(shifts_minutes)
        )
    else:
        shift_spacing = 1.0

    if len(selected_distances) > 1:
        distance_spacing = np.median(
            np.diff(selected_distances)
        )
    else:
        distance_spacing = 1.0

    im = ax_map.imshow(
        correlation_map,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap=CORRELATION_CMAP,
        vmin=CORRELATION_VMIN,
        vmax=CORRELATION_VMAX,
        extent=[
            shifts_minutes[0]
            - shift_spacing / 2.0,
            shifts_minutes[-1]
            + shift_spacing / 2.0,
            selected_distances[0]
            - distance_spacing / 2.0,
            selected_distances[-1]
            + distance_spacing / 2.0,
        ],
    )

    cbar = fig.colorbar(
        im,
        ax=ax_map,
        fraction=0.025,
        pad=0.02,
    )

    cbar.set_label(
        "Pearson correlation with 0 km row"
    )

    ax_map.axvline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.2,
        label="Zero shift",
    )

    ax_map.axhline(
        zero_km_distance,
        color="white",
        linestyle=":",
        linewidth=1.5,
        label=f"Profile row ({zero_km_distance:g} km)",
    )

    ridge_mask = (
        np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (
            maximum_correlation
            >= MIN_CORRELATION_FOR_SLOPE
        )
    )

    if np.any(ridge_mask):
        ax_map.plot(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            color="black",
            linewidth=1.2,
            label="Maximum correlation",
        )

        ax_map.scatter(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            color="black",
            s=8,
        )

    slope = slope_results[
        "slope_min_per_km"
    ]

    intercept = slope_results[
        "intercept_minutes"
    ]

    r_squared = slope_results[
        "r_squared"
    ]

    speed_m_per_s = slope_results[
        "speed_m_per_s"
    ]

    if np.isfinite(slope):
        fit_distances = np.linspace(
            SLOPE_FIT_DISTANCE_MIN_KM,
            SLOPE_FIT_DISTANCE_MAX_KM,
            300,
        )

        fit_shifts = (
            slope
            * fit_distances
            + intercept
        )

        ax_map.plot(
            fit_shifts,
            fit_distances,
            color="lime",
            linestyle="--",
            linewidth=2.0,
            label="Linear ridge fit",
        )

        fit_text = (
            f"Shift slope = {slope:.5f} min/km\n"
            f"Equivalent speed = {speed_m_per_s:.2f} m/s\n"
            f"$R^2$ = {r_squared:.3f}"
        )

        ax_map.text(
            0.02,
            0.03,
            fit_text,
            transform=ax_map.transAxes,
            ha="left",
            va="bottom",
            fontsize=9,
            bbox={
                "facecolor": "white",
                "edgecolor": "black",
                "alpha": 0.85,
            },
        )

    ax_map.set_xlim(
        SHIFT_MIN_MINUTES,
        SHIFT_MAX_MINUTES,
    )

    ax_map.set_ylim(
        CORRELATION_DISTANCE_MIN_KM,
        CORRELATION_DISTANCE_MAX_KM,
    )

    ax_map.set_title(
        f"{angle_deg}° Shifted Row Correlation\n"
        f"{dataset_name}"
    )

    ax_map.set_xlabel(
        "Shift applied to row (min)"
    )

    ax_map.set_ylabel(
        "Distance along slice (km)"
    )

    ax_map.legend(
        loc="upper right",
        fontsize=8,
    )

    # ==================================================
    # 0 km correlation-shift line
    # ==================================================
    ax_profile.plot(
        shifts_minutes,
        zero_km_correlation_profile,
        color="black",
        linewidth=1.4,
        label=f"{zero_km_distance:g} km correlation profile",
    )

    ax_profile.axvline(
        0.0,
        color="gray",
        linestyle="--",
        linewidth=1.0,
    )

    left_peak = zero_km_local_maxima[
        "left"
    ]

    right_peak = zero_km_local_maxima[
        "right"
    ]

    if np.isfinite(
        left_peak["shift_minutes"]
    ):
        ax_profile.scatter(
            left_peak["shift_minutes"],
            left_peak["correlation"],
            s=55,
            marker="o",
            zorder=5,
            label=(
                "Left local max: "
                f"{left_peak['shift_minutes']:.2f} min"
            ),
        )

        ax_profile.axvline(
            left_peak["shift_minutes"],
            linestyle=":",
            linewidth=1.2,
        )

        ax_profile.annotate(
            (
                f"{left_peak['shift_minutes']:.2f} min\n"
                f"r={left_peak['correlation']:.3f}"
            ),
            xy=(
                left_peak["shift_minutes"],
                left_peak["correlation"],
            ),
            xytext=(8, -25),
            textcoords="offset points",
            ha="left",
            va="bottom",
            fontsize=8,
            arrowprops={
                "arrowstyle": "->",
                "linewidth": 0.8,
            },
        )

    if np.isfinite(
        right_peak["shift_minutes"]
    ):
        ax_profile.scatter(
            right_peak["shift_minutes"],
            right_peak["correlation"],
            s=55,
            marker="s",
            zorder=5,
            label=(
                "Right local max: "
                f"{right_peak['shift_minutes']:.2f} min"
            ),
        )

        ax_profile.axvline(
            right_peak["shift_minutes"],
            linestyle=":",
            linewidth=1.2,
        )

        ax_profile.annotate(
            (
                f"{right_peak['shift_minutes']:.2f} min\n"
                f"r={right_peak['correlation']:.3f}"
            ),
            xy=(
                right_peak["shift_minutes"],
                right_peak["correlation"],
            ),
            xytext=(-8, -25),
            textcoords="offset points",
            ha="right",
            va="bottom",
            fontsize=8,
            arrowprops={
                "arrowstyle": "->",
                "linewidth": 0.8,
            },
        )

    ax_profile.set_xlim(
        SHIFT_MIN_MINUTES,
        SHIFT_MAX_MINUTES,
    )

    ax_profile.set_xlabel(
        "Shift applied to row (min)"
    )

    ax_profile.set_ylabel(
        "Correlation"
    )

    ax_profile.set_title(
        f"Correlation-Shift Line at {zero_km_distance:g} km"
    )

    ax_profile.grid(
        True,
        alpha=0.3,
    )

    ax_profile.legend(
        loc="best",
        fontsize=8,
    )

    plt.tight_layout()

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig)


# ==================================================
# Print slope results
# ==================================================
def print_slope_results(
    angle_deg,
    label,
    results,
):
    print(
        f"\n{angle_deg}° — {label}:"
    )

    print(
        f"  Shift slope: "
        f"{results['slope_min_per_km']:.8f} min/km"
    )

    print(
        f"  Intercept: "
        f"{results['intercept_minutes']:.6f} min"
    )

    print(
        f"  Equivalent speed: "
        f"{results['speed_m_per_s']:.4f} m/s"
    )

    print(
        f"  R²: "
        f"{results['r_squared']:.6f}"
    )

    print(
        f"  Points used: "
        f"{results['number_of_fit_points']}"
    )


# ==================================================
# Output paths
# ==================================================
sigma_t_label = safe_number_label(
    SMOOTH_SIGMA_TIME_PIXELS
)

sigma_d_label = safe_number_label(
    SMOOTH_SIGMA_DISTANCE_PIXELS
)

distance_folder = (
    f"r_"
    f"{int(CORRELATION_DISTANCE_MIN_KM)}_"
    f"{int(CORRELATION_DISTANCE_MAX_KM)}"
)

shift_folder = (
    f"shift_min_"
    f"{safe_number_label(SHIFT_MIN_MINUTES)}_"
    f"{safe_number_label(SHIFT_MAX_MINUTES)}"
)

if APPLY_2D_SMOOTHING:
    smoothing_folder = (
        f"sig_"
        f"{sigma_t_label}_"
        f"{sigma_d_label}"
    )
else:
    smoothing_folder = "raw"

OUT_DIR = (
    BASE_OUT_DIR
    / distance_folder
    / smoothing_folder
    / "pchip_bias"
    / "row_correlation_minutes_all_angles"
    / shift_folder
)

UNCORRECTED_OUT_DIR = (
    OUT_DIR
    / "uncorrected"
)

CORRECTED_OUT_DIR = (
    OUT_DIR
    / "corrected"
)

SLOPE_REPORT_OUT_DIR = (
    OUT_DIR
    / "slope_reports"
)

for directory in [
    OUT_DIR,
    UNCORRECTED_OUT_DIR,
    CORRECTED_OUT_DIR,
    SLOPE_REPORT_OUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==================================================
# Read common time array
# ==================================================
TIMES_FILE = (
    BASE_OUT_DIR
    / "times_s.npy"
)

times_s = np.load(
    TIMES_FILE
).astype(float)

times_min = times_s / 60.0

if len(times_min) < 2:
    raise ValueError(
        "At least two time samples are required."
    )


# ==================================================
# Determine minutes per time pixel
# ==================================================
time_differences_min = np.diff(
    times_min
)

finite_time_differences = time_differences_min[
    np.isfinite(time_differences_min)
]

if len(finite_time_differences) == 0:
    raise ValueError(
        "Could not determine the time spacing."
    )

MINUTES_PER_PIXEL = np.median(
    finite_time_differences
)

if (
    not np.isfinite(MINUTES_PER_PIXEL)
    or MINUTES_PER_PIXEL <= 0
):
    raise ValueError(
        "The calculated minutes-per-pixel value is invalid."
    )

maximum_time_spacing_deviation = np.max(
    np.abs(
        finite_time_differences
        - MINUTES_PER_PIXEL
    )
)

print(
    "Angles to process:"
)

print(
    KEOGRAM_ANGLES_DEG
)

print(
    f"Number of angles: "
    f"{len(KEOGRAM_ANGLES_DEG)}"
)

print(
    "\nTime sampling:"
)

print(
    f"One time pixel = "
    f"{MINUTES_PER_PIXEL:.6f} minutes"
)

print(
    f"One time pixel = "
    f"{MINUTES_PER_PIXEL * 60.0:.3f} seconds"
)

print(
    f"Maximum spacing deviation = "
    f"{maximum_time_spacing_deviation:.6e} minutes"
)


# ==================================================
# Convert minute shifts to integer pixel shifts
# ==================================================
shift_min_pixels = int(
    np.ceil(
        SHIFT_MIN_MINUTES
        / MINUTES_PER_PIXEL
    )
)

shift_max_pixels = int(
    np.floor(
        SHIFT_MAX_MINUTES
        / MINUTES_PER_PIXEL
    )
)

shifts_pixels = np.arange(
    shift_min_pixels,
    shift_max_pixels + 1,
    dtype=int,
)

shifts_minutes = (
    shifts_pixels
    * MINUTES_PER_PIXEL
)

print(
    "\nShift range:"
)

print(
    f"Requested: "
    f"{SHIFT_MIN_MINUTES:.2f} to "
    f"{SHIFT_MAX_MINUTES:.2f} min"
)

print(
    f"Actual: "
    f"{shifts_minutes[0]:.2f} to "
    f"{shifts_minutes[-1]:.2f} min"
)

print(
    f"Pixel shifts: "
    f"{shifts_pixels[0]} to "
    f"{shifts_pixels[-1]}"
)


# ==================================================
# Select common time range
# ==================================================
time_mask = (
    (times_min >= CORRELATION_TIME_MIN)
    & (times_min <= CORRELATION_TIME_MAX)
)

if np.count_nonzero(time_mask) < MIN_VALID_CORRELATION_POINTS:
    raise ValueError(
        "The selected time interval has too few samples."
    )

times_correlation_min = times_min[
    time_mask
]


# ==================================================
# Save common arrays
# ==================================================
np.save(
    OUT_DIR / "angles_deg.npy",
    KEOGRAM_ANGLES_DEG,
)

np.save(
    OUT_DIR / "shifts_minutes.npy",
    shifts_minutes,
)

np.save(
    OUT_DIR / "shifts_pixels.npy",
    shifts_pixels,
)

np.save(
    OUT_DIR / "correlation_times_min.npy",
    times_correlation_min,
)


# ==================================================
# Storage for combined angle results
# ==================================================
combined_results = []
combined_zero_km_peak_results = []

successful_angles = []
failed_angles = []


# ==================================================
# Process every angle
# ==================================================
for angle_number, angle_deg in enumerate(
    KEOGRAM_ANGLES_DEG,
    start=1,
):
    print(
        "\n"
        + "=" * 70
    )

    print(
        f"Processing angle "
        f"{angle_deg}° "
        f"({angle_number}/{len(KEOGRAM_ANGLES_DEG)})"
    )

    print(
        "=" * 70
    )

    KEOGRAM_FILE = (
        BASE_OUT_DIR
        / f"BLO_keogram_angle_{angle_deg:03d}.npy"
    )

    if not KEOGRAM_FILE.exists():
        print(
            f"WARNING: File not found: "
            f"{KEOGRAM_FILE}"
        )

        failed_angles.append(
            (
                int(angle_deg),
                "file not found",
            )
        )

        continue

    try:
        # ==================================================
        # Read keogram
        # ==================================================
        keogram_original = np.load(
            KEOGRAM_FILE
        ).astype(float)

        if keogram_original.ndim != 2:
            raise ValueError(
                f"Expected a 2D keogram, found "
                f"{keogram_original.shape}."
            )

        nt, nspace = keogram_original.shape

        if len(times_min) != nt:
            raise ValueError(
                f"Keogram has {nt} time samples, but "
                f"times_s.npy has {len(times_min)}."
            )

        # ==================================================
        # Construct distance axis
        # ==================================================
        distance_km = np.arange(
            -(nspace // 2),
            nspace // 2 + 1,
            dtype=float,
        )

        if len(distance_km) != nspace:
            distance_km = (
                np.arange(
                    nspace,
                    dtype=float,
                )
                - nspace // 2
            )

        available_distance_min = np.nanmin(
            distance_km
        )

        available_distance_max = np.nanmax(
            distance_km
        )

        if (
            CORRELATION_DISTANCE_MIN_KM
            < available_distance_min
            or CORRELATION_DISTANCE_MAX_KM
            > available_distance_max
        ):
            raise ValueError(
                f"Requested distance range "
                f"{CORRELATION_DISTANCE_MIN_KM} to "
                f"{CORRELATION_DISTANCE_MAX_KM} km "
                f"is outside available range "
                f"{available_distance_min} to "
                f"{available_distance_max} km."
            )

        # ==================================================
        # Select time interval
        # ==================================================
        keogram_time_selected = keogram_original[
            time_mask,
            :
        ]

        # ==================================================
        # Optional smoothing
        # ==================================================
        if APPLY_2D_SMOOTHING:
            keogram_uncorrected = smooth_keogram_2d(
                data=keogram_time_selected,
                sigma_time=SMOOTH_SIGMA_TIME_PIXELS,
                sigma_distance=SMOOTH_SIGMA_DISTANCE_PIXELS,
                mode=SMOOTH_MODE,
                weight_tolerance=SMOOTH_WEIGHT_TOLERANCE,
            )

            uncorrected_dataset_name = (
                f"Uncorrected, smoothed "
                f"σt={SMOOTH_SIGMA_TIME_PIXELS:g}, "
                f"σd={SMOOTH_SIGMA_DISTANCE_PIXELS:g}"
            )
        else:
            keogram_uncorrected = (
                keogram_time_selected.copy()
            )

            uncorrected_dataset_name = (
                "Uncorrected, unsmoothed"
            )

        # ==================================================
        # Create bias correction
        # ==================================================
        (
            bias_correction_keogram,
            bias_vs_time,
            bias_distance_factor,
        ) = create_bias_correction_keogram(
            times_min=times_correlation_min,
            distance_km=distance_km,
            bias_time_points_min=BIAS_TIME_POINTS_MIN,
            bias_at_max_distance=BIAS_AT_MAX_DISTANCE,
            bias_distance_min_km=BIAS_DISTANCE_MIN_KM,
            bias_distance_max_km=BIAS_DISTANCE_MAX_KM,
            distance_power=BIAS_DISTANCE_POWER,
            zero_outside_distance_range=(
                BIAS_ZERO_OUTSIDE_DISTANCE_RANGE
            ),
            zero_outside_time_range=(
                BIAS_ZERO_OUTSIDE_TIME_RANGE
            ),
            clip_negative_time_correction=(
                CLIP_NEGATIVE_TIME_CORRECTION
            ),
        )

        # ==================================================
        # Apply bias correction
        # ==================================================
        if APPLY_BIAS_CORRECTION:
            keogram_corrected = (
                keogram_uncorrected
                + bias_correction_keogram
            )

            corrected_dataset_name = (
                "Bias corrected: PCHIP time dependence, "
                f"distance power={BIAS_DISTANCE_POWER:g}"
            )
        else:
            keogram_corrected = (
                keogram_uncorrected.copy()
            )

            corrected_dataset_name = (
                "Bias correction disabled"
            )

        keogram_corrected[
            ~np.isfinite(keogram_uncorrected)
        ] = np.nan

        # ==================================================
        # Select distance rows
        # ==================================================
        selected_distance_mask = (
            (
                distance_km
                >= CORRELATION_DISTANCE_MIN_KM
            )
            & (
                distance_km
                <= CORRELATION_DISTANCE_MAX_KM
            )
        )

        selected_distances = distance_km[
            selected_distance_mask
        ]

        if len(selected_distances) == 0:
            raise ValueError(
                "No rows lie inside the selected distance range."
            )

        # ==================================================
        # Per-angle folders
        # ==================================================
        angle_uncorrected_dir = (
            UNCORRECTED_OUT_DIR
            / f"angle_{angle_deg:03d}"
        )

        angle_corrected_dir = (
            CORRECTED_OUT_DIR
            / f"angle_{angle_deg:03d}"
        )

        angle_uncorrected_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        angle_corrected_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        # ==================================================
        # Uncorrected map
        # ==================================================
        (
            correlation_map_uncorrected,
            selected_distances_uncorrected,
            reference_index_uncorrected,
        ) = calculate_correlation_map(
            keogram=keogram_uncorrected,
            distance_km=distance_km,
            reference_distance_km=0.0,
            selected_distance_mask=selected_distance_mask,
            shifts_pixels=shifts_pixels,
            minimum_points=MIN_VALID_CORRELATION_POINTS,
        )

        (
            best_shift_uncorrected_minutes,
            maximum_correlation_uncorrected,
        ) = find_best_shifts(
            correlation_map=correlation_map_uncorrected,
            shifts_minutes=shifts_minutes,
        )

        uncorrected_slope_results = (
            fit_maximum_correlation_slope(
                distances_km=(
                    selected_distances_uncorrected
                ),
                best_shift_minutes=(
                    best_shift_uncorrected_minutes
                ),
                maximum_correlation=(
                    maximum_correlation_uncorrected
                ),
                minimum_correlation=(
                    MIN_CORRELATION_FOR_SLOPE
                ),
                distance_min_km=(
                    SLOPE_FIT_DISTANCE_MIN_KM
                ),
                distance_max_km=(
                    SLOPE_FIT_DISTANCE_MAX_KM
                ),
            )
        )

        # ==================================================
        # Corrected map
        # ==================================================
        (
            correlation_map_corrected,
            selected_distances_corrected,
            reference_index_corrected,
        ) = calculate_correlation_map(
            keogram=keogram_corrected,
            distance_km=distance_km,
            reference_distance_km=0.0,
            selected_distance_mask=selected_distance_mask,
            shifts_pixels=shifts_pixels,
            minimum_points=MIN_VALID_CORRELATION_POINTS,
        )

        (
            best_shift_corrected_minutes,
            maximum_correlation_corrected,
        ) = find_best_shifts(
            correlation_map=correlation_map_corrected,
            shifts_minutes=shifts_minutes,
        )

        corrected_slope_results = (
            fit_maximum_correlation_slope(
                distances_km=(
                    selected_distances_corrected
                ),
                best_shift_minutes=(
                    best_shift_corrected_minutes
                ),
                maximum_correlation=(
                    maximum_correlation_corrected
                ),
                minimum_correlation=(
                    MIN_CORRELATION_FOR_SLOPE
                ),
                distance_min_km=(
                    SLOPE_FIT_DISTANCE_MIN_KM
                ),
                distance_max_km=(
                    SLOPE_FIT_DISTANCE_MAX_KM
                ),
            )
        )

        # ==================================================
        # Extract 0 km correlation-shift profiles
        # ==================================================
        (
            zero_km_profile_uncorrected,
            zero_km_row_uncorrected,
            zero_km_distance_uncorrected,
        ) = extract_zero_km_correlation_profile(
            correlation_map=correlation_map_uncorrected,
            selected_distances=selected_distances_uncorrected,
        )

        (
            zero_km_profile_corrected,
            zero_km_row_corrected,
            zero_km_distance_corrected,
        ) = extract_zero_km_correlation_profile(
            correlation_map=correlation_map_corrected,
            selected_distances=selected_distances_corrected,
        )

        zero_km_peaks_uncorrected = (
            find_left_and_right_local_maxima(
                correlation_profile=zero_km_profile_uncorrected,
                shifts_minutes=shifts_minutes,
                minimum_prominence=(
                    LOCAL_MAXIMUM_MIN_PROMINENCE
                ),
                minimum_distance_pixels=(
                    LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS
                ),
            )
        )

        zero_km_peaks_corrected = (
            find_left_and_right_local_maxima(
                correlation_profile=zero_km_profile_corrected,
                shifts_minutes=shifts_minutes,
                minimum_prominence=(
                    LOCAL_MAXIMUM_MIN_PROMINENCE
                ),
                minimum_distance_pixels=(
                    LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS
                ),
            )
        )

        # ==================================================
        # Plot maps
        # ==================================================
        uncorrected_plot_file = (
            angle_uncorrected_dir
            / (
                f"correlation_minutes_"
                f"{angle_deg:03d}_uncorrected.png"
            )
        )

        corrected_plot_file = (
            angle_corrected_dir
            / (
                f"correlation_minutes_"
                f"{angle_deg:03d}_corrected.png"
            )
        )

        plot_correlation_map(
            angle_deg=angle_deg,
            full_keogram=keogram_original,
            full_times_min=times_min,
            full_distance_km=distance_km,
            limited_keogram=keogram_uncorrected[
                :,
                selected_distance_mask,
            ],
            limited_times_min=times_correlation_min,
            limited_distance_km=selected_distances_uncorrected,
            correlation_map=(
                correlation_map_uncorrected
            ),
            shifts_minutes=shifts_minutes,
            selected_distances=(
                selected_distances_uncorrected
            ),
            best_shift_minutes=(
                best_shift_uncorrected_minutes
            ),
            maximum_correlation=(
                maximum_correlation_uncorrected
            ),
            slope_results=(
                uncorrected_slope_results
            ),
            zero_km_correlation_profile=(
                zero_km_profile_uncorrected
            ),
            zero_km_distance=(
                zero_km_distance_uncorrected
            ),
            zero_km_local_maxima=(
                zero_km_peaks_uncorrected
            ),
            dataset_name=uncorrected_dataset_name,
            output_file=uncorrected_plot_file,
        )

        plot_correlation_map(
            angle_deg=angle_deg,
            full_keogram=keogram_original,
            full_times_min=times_min,
            full_distance_km=distance_km,
            limited_keogram=keogram_corrected[
                :,
                selected_distance_mask,
            ],
            limited_times_min=times_correlation_min,
            limited_distance_km=selected_distances_corrected,
            correlation_map=(
                correlation_map_corrected
            ),
            shifts_minutes=shifts_minutes,
            selected_distances=(
                selected_distances_corrected
            ),
            best_shift_minutes=(
                best_shift_corrected_minutes
            ),
            maximum_correlation=(
                maximum_correlation_corrected
            ),
            slope_results=(
                corrected_slope_results
            ),
            zero_km_correlation_profile=(
                zero_km_profile_corrected
            ),
            zero_km_distance=(
                zero_km_distance_corrected
            ),
            zero_km_local_maxima=(
                zero_km_peaks_corrected
            ),
            dataset_name=corrected_dataset_name,
            output_file=corrected_plot_file,
        )

        # ==================================================
        # Save arrays
        # ==================================================
        np.save(
            angle_uncorrected_dir
            / (
                f"correlation_"
                f"{angle_deg:03d}_uncorrected.npy"
            ),
            correlation_map_uncorrected,
        )

        np.save(
            angle_uncorrected_dir
            / (
                f"best_shift_minutes_"
                f"{angle_deg:03d}_uncorrected.npy"
            ),
            best_shift_uncorrected_minutes,
        )

        np.save(
            angle_uncorrected_dir
            / (
                f"maximum_correlation_"
                f"{angle_deg:03d}_uncorrected.npy"
            ),
            maximum_correlation_uncorrected,
        )

        np.save(
            angle_corrected_dir
            / (
                f"correlation_"
                f"{angle_deg:03d}_corrected.npy"
            ),
            correlation_map_corrected,
        )

        np.save(
            angle_corrected_dir
            / (
                f"best_shift_minutes_"
                f"{angle_deg:03d}_corrected.npy"
            ),
            best_shift_corrected_minutes,
        )

        np.save(
            angle_corrected_dir
            / (
                f"maximum_correlation_"
                f"{angle_deg:03d}_corrected.npy"
            ),
            maximum_correlation_corrected,
        )

        np.save(
            angle_uncorrected_dir
            / "distances_km.npy",
            selected_distances_uncorrected,
        )

        np.save(
            angle_corrected_dir
            / "distances_km.npy",
            selected_distances_corrected,
        )

        np.save(
            angle_uncorrected_dir
            / "zero_km_correlation_profile.npy",
            zero_km_profile_uncorrected,
        )

        np.save(
            angle_corrected_dir
            / "zero_km_correlation_profile.npy",
            zero_km_profile_corrected,
        )

        zero_km_profile_uncorrected_csv = np.column_stack(
            [
                shifts_minutes,
                zero_km_profile_uncorrected,
            ]
        )

        zero_km_profile_corrected_csv = np.column_stack(
            [
                shifts_minutes,
                zero_km_profile_corrected,
            ]
        )

        np.savetxt(
            angle_uncorrected_dir
            / "zero_km_correlation_profile.csv",
            zero_km_profile_uncorrected_csv,
            delimiter=",",
            header="shift_minutes,correlation",
            comments="",
        )

        np.savetxt(
            angle_corrected_dir
            / "zero_km_correlation_profile.csv",
            zero_km_profile_corrected_csv,
            delimiter=",",
            header="shift_minutes,correlation",
            comments="",
        )

        # ==================================================
        # Save ridge CSV files
        # ==================================================
        uncorrected_summary = np.column_stack(
            [
                selected_distances_uncorrected,
                best_shift_uncorrected_minutes,
                maximum_correlation_uncorrected,
            ]
        )

        corrected_summary = np.column_stack(
            [
                selected_distances_corrected,
                best_shift_corrected_minutes,
                maximum_correlation_corrected,
            ]
        )

        uncorrected_csv = (
            angle_uncorrected_dir
            / (
                f"ridge_"
                f"{angle_deg:03d}_uncorrected.csv"
            )
        )

        corrected_csv = (
            angle_corrected_dir
            / (
                f"ridge_"
                f"{angle_deg:03d}_corrected.csv"
            )
        )

        np.savetxt(
            uncorrected_csv,
            uncorrected_summary,
            delimiter=",",
            header=(
                "distance_km,"
                "best_shift_minutes,"
                "maximum_correlation"
            ),
            comments="",
        )

        np.savetxt(
            corrected_csv,
            corrected_summary,
            delimiter=",",
            header=(
                "distance_km,"
                "best_shift_minutes,"
                "maximum_correlation"
            ),
            comments="",
        )

        # ==================================================
        # Save individual slope report
        # ==================================================
        slope_output_file = (
            SLOPE_REPORT_OUT_DIR
            / (
                f"correlation_slope_"
                f"{angle_deg:03d}.txt"
            )
        )

        with open(
            slope_output_file,
            "w",
            encoding="utf-8",
        ) as file:
            file.write(
                f"Angle: {angle_deg} degrees\n"
            )

            file.write(
                f"Minutes per pixel: "
                f"{MINUTES_PER_PIXEL:.10f}\n\n"
            )

            file.write(
                "UNCORRECTED\n"
            )

            for key, value in (
                uncorrected_slope_results.items()
            ):
                if key != "fit_mask":
                    file.write(
                        f"{key}: {value}\n"
                    )

            file.write(
                "\nCORRECTED\n"
            )

            for key, value in (
                corrected_slope_results.items()
            ):
                if key != "fit_mask":
                    file.write(
                        f"{key}: {value}\n"
                    )

            file.write(
                "\n0 KM CORRELATION-PROFILE LOCAL MAXIMA\n"
            )

            file.write(
                "Uncorrected left shift (min): "
                f"{zero_km_peaks_uncorrected['left']['shift_minutes']}\n"
            )
            file.write(
                "Uncorrected left correlation: "
                f"{zero_km_peaks_uncorrected['left']['correlation']}\n"
            )
            file.write(
                "Uncorrected right shift (min): "
                f"{zero_km_peaks_uncorrected['right']['shift_minutes']}\n"
            )
            file.write(
                "Uncorrected right correlation: "
                f"{zero_km_peaks_uncorrected['right']['correlation']}\n"
            )
            file.write(
                "Corrected left shift (min): "
                f"{zero_km_peaks_corrected['left']['shift_minutes']}\n"
            )
            file.write(
                "Corrected left correlation: "
                f"{zero_km_peaks_corrected['left']['correlation']}\n"
            )
            file.write(
                "Corrected right shift (min): "
                f"{zero_km_peaks_corrected['right']['shift_minutes']}\n"
            )
            file.write(
                "Corrected right correlation: "
                f"{zero_km_peaks_corrected['right']['correlation']}\n"
            )

        # ==================================================
        # Add to combined summary
        # ==================================================
        combined_results.append(
            [
                float(angle_deg),

                uncorrected_slope_results[
                    "slope_min_per_km"
                ],
                uncorrected_slope_results[
                    "intercept_minutes"
                ],
                uncorrected_slope_results[
                    "speed_km_per_min"
                ],
                uncorrected_slope_results[
                    "speed_m_per_s"
                ],
                uncorrected_slope_results[
                    "r_squared"
                ],
                uncorrected_slope_results[
                    "number_of_fit_points"
                ],

                corrected_slope_results[
                    "slope_min_per_km"
                ],
                corrected_slope_results[
                    "intercept_minutes"
                ],
                corrected_slope_results[
                    "speed_km_per_min"
                ],
                corrected_slope_results[
                    "speed_m_per_s"
                ],
                corrected_slope_results[
                    "r_squared"
                ],
                corrected_slope_results[
                    "number_of_fit_points"
                ],
            ]
        )

        combined_zero_km_peak_results.append(
            [
                float(angle_deg),
                zero_km_peaks_uncorrected["left"][
                    "shift_minutes"
                ],
                zero_km_peaks_uncorrected["left"][
                    "correlation"
                ],
                zero_km_peaks_uncorrected["right"][
                    "shift_minutes"
                ],
                zero_km_peaks_uncorrected["right"][
                    "correlation"
                ],
                zero_km_peaks_corrected["left"][
                    "shift_minutes"
                ],
                zero_km_peaks_corrected["left"][
                    "correlation"
                ],
                zero_km_peaks_corrected["right"][
                    "shift_minutes"
                ],
                zero_km_peaks_corrected["right"][
                    "correlation"
                ],
            ]
        )

        successful_angles.append(
            int(angle_deg)
        )

        print_slope_results(
            angle_deg=angle_deg,
            label=(
                "Uncorrected maximum-correlation ridge"
            ),
            results=uncorrected_slope_results,
        )

        print_slope_results(
            angle_deg=angle_deg,
            label=(
                "Corrected maximum-correlation ridge"
            ),
            results=corrected_slope_results,
        )

        print(
            "\n0 km correlation-profile local maxima:"
        )
        print(
            "  Uncorrected left: "
            f"{zero_km_peaks_uncorrected['left']['shift_minutes']:.3f} min, "
            f"r={zero_km_peaks_uncorrected['left']['correlation']:.6f}"
        )
        print(
            "  Uncorrected right: "
            f"{zero_km_peaks_uncorrected['right']['shift_minutes']:.3f} min, "
            f"r={zero_km_peaks_uncorrected['right']['correlation']:.6f}"
        )
        print(
            "  Corrected left: "
            f"{zero_km_peaks_corrected['left']['shift_minutes']:.3f} min, "
            f"r={zero_km_peaks_corrected['left']['correlation']:.6f}"
        )
        print(
            "  Corrected right: "
            f"{zero_km_peaks_corrected['right']['shift_minutes']:.3f} min, "
            f"r={zero_km_peaks_corrected['right']['correlation']:.6f}"
        )

    except Exception as error:
        print(
            f"ERROR processing {angle_deg}°:"
        )

        print(
            error
        )

        failed_angles.append(
            (
                int(angle_deg),
                str(error),
            )
        )


# ==================================================
# Save combined 0 km local-maximum summary
# ==================================================
combined_zero_km_peak_results = np.asarray(
    combined_zero_km_peak_results,
    dtype=float,
)

combined_zero_km_peak_csv = (
    OUT_DIR
    / "all_angle_zero_km_local_maxima.csv"
)

combined_zero_km_peak_npy = (
    OUT_DIR
    / "all_angle_zero_km_local_maxima.npy"
)

combined_zero_km_peak_header = (
    "angle_deg,"
    "uncorrected_left_shift_minutes,"
    "uncorrected_left_correlation,"
    "uncorrected_right_shift_minutes,"
    "uncorrected_right_correlation,"
    "corrected_left_shift_minutes,"
    "corrected_left_correlation,"
    "corrected_right_shift_minutes,"
    "corrected_right_correlation"
)

if combined_zero_km_peak_results.size > 0:
    np.savetxt(
        combined_zero_km_peak_csv,
        combined_zero_km_peak_results,
        delimiter=",",
        header=combined_zero_km_peak_header,
        comments="",
    )

    np.save(
        combined_zero_km_peak_npy,
        combined_zero_km_peak_results,
    )


# ==================================================
# Save combined angle summary
# ==================================================
combined_results = np.asarray(
    combined_results,
    dtype=float,
)

combined_csv = (
    OUT_DIR
    / "all_angle_slope_summary.csv"
)

combined_npy = (
    OUT_DIR
    / "all_angle_slope_summary.npy"
)

combined_header = (
    "angle_deg,"
    "uncorrected_slope_min_per_km,"
    "uncorrected_intercept_minutes,"
    "uncorrected_speed_km_per_min,"
    "uncorrected_speed_m_per_s,"
    "uncorrected_r_squared,"
    "uncorrected_number_of_fit_points,"
    "corrected_slope_min_per_km,"
    "corrected_intercept_minutes,"
    "corrected_speed_km_per_min,"
    "corrected_speed_m_per_s,"
    "corrected_r_squared,"
    "corrected_number_of_fit_points"
)

if combined_results.size > 0:
    np.savetxt(
        combined_csv,
        combined_results,
        delimiter=",",
        header=combined_header,
        comments="",
    )

    np.save(
        combined_npy,
        combined_results,
    )


# ==================================================
# Plot speed versus angle
# ==================================================
if combined_results.size > 0:
    angles_processed = combined_results[
        :,
        0,
    ]

    uncorrected_speed_m_per_s = combined_results[
        :,
        4,
    ]

    corrected_speed_m_per_s = combined_results[
        :,
        10,
    ]

    fig_speed, ax_speed = plt.subplots(
        figsize=(10, 5)
    )

    ax_speed.plot(
        angles_processed,
        uncorrected_speed_m_per_s,
        marker="o",
        linewidth=1.5,
        label="Uncorrected",
    )

    ax_speed.plot(
        angles_processed,
        corrected_speed_m_per_s,
        marker="o",
        linewidth=1.5,
        label="Corrected",
    )

    ax_speed.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
    )

    ax_speed.set_xlim(
        KEOGRAM_ANGLES_DEG.min(),
        KEOGRAM_ANGLES_DEG.max(),
    )

    ax_speed.set_xlabel(
        "Keogram angle (degrees)"
    )

    ax_speed.set_ylabel(
        "Equivalent signed speed (m/s)"
    )

    ax_speed.set_title(
        "Maximum-Correlation Ridge Speed versus Angle"
    )

    ax_speed.grid(
        True,
        alpha=0.3,
    )

    ax_speed.legend(
        loc="best",
    )

    plt.tight_layout()

    speed_angle_plot = (
        OUT_DIR
        / "speed_vs_angle.png"
    )

    fig_speed.savefig(
        speed_angle_plot,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig_speed
    )


# ==================================================
# Plot R-squared versus angle
# ==================================================
if combined_results.size > 0:
    uncorrected_r_squared = combined_results[
        :,
        5,
    ]

    corrected_r_squared = combined_results[
        :,
        11,
    ]

    fig_r2, ax_r2 = plt.subplots(
        figsize=(10, 5)
    )

    ax_r2.plot(
        angles_processed,
        uncorrected_r_squared,
        marker="o",
        linewidth=1.5,
        label="Uncorrected",
    )

    ax_r2.plot(
        angles_processed,
        corrected_r_squared,
        marker="o",
        linewidth=1.5,
        label="Corrected",
    )

    ax_r2.set_xlim(
        KEOGRAM_ANGLES_DEG.min(),
        KEOGRAM_ANGLES_DEG.max(),
    )

    ax_r2.set_ylim(
        0.0,
        1.0,
    )

    ax_r2.set_xlabel(
        "Keogram angle (degrees)"
    )

    ax_r2.set_ylabel(
        "R²"
    )

    ax_r2.set_title(
        "Maximum-Correlation Ridge Fit R² versus Angle"
    )

    ax_r2.grid(
        True,
        alpha=0.3,
    )

    ax_r2.legend(
        loc="best",
    )

    plt.tight_layout()

    r2_angle_plot = (
        OUT_DIR
        / "r_squared_vs_angle.png"
    )

    fig_r2.savefig(
        r2_angle_plot,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig_r2
    )


# ==================================================
# Save processing log
# ==================================================
processing_log_file = (
    OUT_DIR
    / "processing_log.txt"
)

with open(
    processing_log_file,
    "w",
    encoding="utf-8",
) as file:
    file.write(
        "Requested angles:\n"
    )

    file.write(
        ", ".join(
            str(angle)
            for angle in KEOGRAM_ANGLES_DEG
        )
        + "\n\n"
    )

    file.write(
        f"Number requested: "
        f"{len(KEOGRAM_ANGLES_DEG)}\n"
    )

    file.write(
        f"Number successful: "
        f"{len(successful_angles)}\n"
    )

    file.write(
        f"Number failed: "
        f"{len(failed_angles)}\n\n"
    )

    file.write(
        "Successful angles:\n"
    )

    file.write(
        ", ".join(
            str(angle)
            for angle in successful_angles
        )
        + "\n\n"
    )

    file.write(
        "Failed angles:\n"
    )

    if failed_angles:
        for angle_deg, reason in failed_angles:
            file.write(
                f"{angle_deg}: {reason}\n"
            )
    else:
        file.write(
            "None\n"
        )


# ==================================================
# Print final summary
# ==================================================
print(
    "\n"
    + "=" * 70
)

print(
    "ALL-ANGLE PROCESSING COMPLETE"
)

print(
    "=" * 70
)

print(
    "Angles requested:"
)

print(
    KEOGRAM_ANGLES_DEG
)

print(
    f"\nNumber requested: "
    f"{len(KEOGRAM_ANGLES_DEG)}"
)

print(
    f"Number successful: "
    f"{len(successful_angles)}"
)

print(
    f"Number failed: "
    f"{len(failed_angles)}"
)

print(
    "\nSuccessful angles:"
)

print(
    successful_angles
)

if failed_angles:
    print(
        "\nFailed angles:"
    )

    for angle_deg, reason in failed_angles:
        print(
            f"  {angle_deg}°: {reason}"
        )

print(
    "\nCombined CSV:"
)

print(
    combined_csv
)

print(
    "\nOutput folder:"
)

print(
    OUT_DIR.resolve()
)

Angles to process:
[  0   5  10  15  20  25  30  35  40  45  50  55  60  65  70  75  80  85
  90  95 100 105 110 115 120 125 130 135 140 145 150 155 160 165 170 175]
Number of angles: 36

Time sampling:
One time pixel = 2.000000 minutes
One time pixel = 120.000 seconds
Maximum spacing deviation = 0.000000e+00 minutes

Shift range:
Requested: -60.00 to 60.00 min
Actual: -60.00 to 60.00 min
Pixel shifts: -30 to 30

Processing angle 0° (1/36)

0° — Uncorrected maximum-correlation ridge:
  Shift slope: 0.01900057 min/km
  Intercept: 0.979511 min
  Equivalent speed: 877.1667 m/s
  R²: 0.712775
  Points used: 251

0° — Corrected maximum-correlation ridge:
  Shift slope: 0.01900057 min/km
  Intercept: 0.979511 min
  Equivalent speed: 877.1667 m/s
  R²: 0.712775
  Points used: 251

0 km correlation-profile local maxima:
  Uncorrected left: -22.000 min, r=0.880725
  Uncorrected right: 22.000 min, r=0.880725
  Corrected left: -22.000 min, r=0.880725
  Corrected right: 22.000 min, r=0.880725

Pro